# Fitness-AQA — Pose-Based Action Quality Assessment

**Exercises:** BackSquat · BarbellRow · OverheadPress (OHP)  
**Approach:** Per-class binary TCN classifiers — one model per error type, trained on balanced 1:1 datasets.  
**Quality Score:** Derived post-hoc as `score = 1 − mean(error_probabilities)`.


> **Note.** This is the original research notebook from the Imperial College I-Explore project, kept as a record of the experiments. The production code lives in the `exercise_advisor` package (see the repository README); the notebook writes checkpoints in its *legacy* format, which `scripts/convert_legacy_checkpoints.py` converts to the package format. Prediction-example images have been removed from the outputs because they show dataset subjects.

---

### Pipeline Overview

| Cell(s) | Stage | Description |
|---------|-------|-------------|
| 1 | **Config & Imports** | Global settings, paths, hyperparameters |
| 2–3 | **Label Loading & EDA** | Load labels, derive quality scores, visualise class balance |
| 4–8 | **Pose Extraction** | MediaPipe / TorchVision keypoint extraction from video/images |
| 9–10 | **Normalisation & Features** | Hip-centred normalisation, joint angles, velocity features |
| 11–14 | **Dataset & DataLoaders** | Augmentation, balanced binary datasets, DataLoader factories |
| 15 | **Subject-Aware Splits** | Verify no data leakage across train/val/test |
| 16 | **TCN Model** | Dilated causal Conv1D + AttentionPool, classification-only head |
| 17–18 | **Training** | Per-class binary training with early stopping, threshold sweep |
| 19–20 | **Evaluation** | Per-class P/R/F1/AUC, derived score correlation, plots |
| 21 | **Inference** | `AQAPredictor` — end-to-end video-to-feedback pipeline |

### Requirements

```
torch >= 2.0
numpy, scipy, scikit-learn, matplotlib
mediapipe >= 0.10 (for pose extraction only)
opencv-python (for pose extraction only)
```

---
## Configuration & Global Imports

Global paths, hyperparameters, and reproducibility settings.  
Run this cell before any other pipeline cell.

In [ ]:
"""
Global configuration and imports for the Fitness-AQA pipeline.
Run this cell before any other pipeline cell.
"""
import json
import random
import os
import warnings
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_ROOT   = Path(os.environ.get("FITNESS_AQA_ROOT", "../data/Fitness-AQA_dataset_release"))
POSE_ROOT      = Path(os.environ.get("EXERCISE_ADVISOR_POSES", "../data/poses"))        # extracted pose npy files
PROCESSED_ROOT = Path(os.environ.get("EXERCISE_ADVISOR_PROCESSED", "../data/processed"))  # normalised + interpolated
BARBELL_IMAGES_DIR = (
    DATASET_ROOT / "BarbellRow" / "Labeled_Dataset" / "barbellrow_images_raw"
)
for p in [POSE_ROOT, PROCESSED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

def video_dir_for_exercise(exercise: str) -> Path:
    return DATASET_ROOT / exercise / "Labeled_Dataset" / "videos"

print("Video directories:")
for _ex in ["OHP", "Squat", "BarbellRow"]:
    _vd = video_dir_for_exercise(_ex)
    _n  = len(list(_vd.glob("*.mp4"))) if _vd.exists() else 0
    _status = "✓" if _n > 0 else "✗ (not found)"
    print(f"  {_ex:12s}: {_n:5d} videos   [{_vd.relative_to(DATASET_ROOT)}]  {_status}")

# ── Temporal model config ──────────────────────────────────────────────────────
@dataclass
class Config:
    fixed_len:        int   = 100
    n_landmarks:      int   = 33
    n_coords:         int   = 3
    feature_dim:      int   = 99        # computed in __post_init__
    use_angles:       bool  = True
    use_velocity:     bool  = True
    batch_size:       int   = 32
    lr:               float = 3e-4      # gentler LR for stable convergence
    max_epochs:       int   = 200
    patience:         int   = 30        # generous patience
    weight_decay:     float = 5e-4
    pos_weight_scale: float = 1.0
    focal_gamma:      float = 2.0       # focal loss focusing parameter
    label_smoothing:  float = 0.0       # disabled — focal loss handles calibration
    # ── Architecture ───────────────────────────────────────────────────────
    hidden_dim:       int   = 128       # TCN channel width (balanced for data size)
    n_layers:         int   = 6         # TCN depth: RF covers 100 frames
    dropout:          float = 0.3       # moderate dropout

    # ── Per-exercise oversample factors ────────────────────────────────────
    # Higher for exercises with more class imbalance.
    # BarbellRow: lumbar 6.7×, torso 10.7× → needs aggressive oversample
    # Squat: knees_inward 21.6× → needs aggressive oversample
    # OHP: relatively balanced (2–3×) → moderate oversample
    oversample_factors: Dict = None

    def __post_init__(self):
        base     = self.n_landmarks * self.n_coords          # 99
        n_angles = 12 if self.use_angles else 0
        n_vel    = (base + n_angles) if self.use_velocity else 0
        self.feature_dim = base + n_angles + n_vel
        if self.oversample_factors is None:
            self.oversample_factors = {
                "OHP":        3,    # mild imbalance → 3×
                "Squat":      8,    # knees_inward 21.6× imbalance → 8×
                "BarbellRow": 6,    # torso_angle 10.7× imbalance → 6×
            }

cfg = Config()
print(f"\nFeature dim per frame : {cfg.feature_dim}")
print(f"Input tensor shape    : ({cfg.fixed_len}, {cfg.feature_dim})")
print(f"Seed                  : {SEED}")
print(f"Architecture          : hidden={cfg.hidden_dim}, layers={cfg.n_layers}, dropout={cfg.dropout}")
print(f"Learning rate         : {cfg.lr}")
print(f"Weight decay          : {cfg.weight_decay}")
print(f"Patience              : {cfg.patience}")
print(f"focal_gamma           : {cfg.focal_gamma}")
print(f"Oversample factors    : {cfg.oversample_factors}")


---
## Label Loading & Quality Score Derivation

**Three label formats in this dataset:**

| Exercise | Label file | Format | Value |
|---|---|---|---|
| OHP | `error_elbows`, `error_knees` | `interval` | `[[t_start, t_end], ...]` (seconds); empty = no error |
| Squat | `knees_inward`, `knees_forward` | `interval` | `[[t_start, t_end], ...]` (seconds); empty = no error |
| Squat | `shallow_depth` | `frame_binary` | `0`/`1` per frame; aggregated to rep-level (any frame=1 → rep=1) |
| BarbellRow | `lumbar_error`, `torso_angle` | `binary` | `0` = no error, `1` = error |

**Key formats:**
- OHP / Squat: `{subject_id}_{rep_number}` (e.g. `46777_3`)
- BarbellRow: `{subject_id}_{session}_{rep}` (e.g. `56067_3_51`)

**Quality score derivation:**
- Interval labels → `score = 1 − clamp(total_error_seconds / clip_duration, 0, 1)`
- Binary labels → `score = 1.0` if clean, `0.0` if any error present
- Mixed (Squat) → interval score penalised by binary errors

In [ ]:
"""
Label loading utilities for all three exercises.
Handles three label formats found in this dataset:
  - 'interval'      : [[t_start, t_end], ...]  (OHP, Squat knees errors)
  - 'binary'        : 0 or 1                   (BarbellRow)
  - 'frame_binary'  : 0 or 1 with 3-part keys  (Squat shallow_depth)
      Keys are subject_rep_frame (e.g. 37803_2_44 = frame 44 of rep 37803_2).
      Aggregated to rep-level: if ANY frame has label=1 → rep=1.
"""
from dataclasses import dataclass
from typing import List, Dict
import numpy as np, json
from pathlib import Path

# ── Exercise definitions ───────────────────────────────────────────────────────
# label_format: 'interval' → temporal [[t0,t1],...] or 'binary' → 0/1 int
EXERCISE_CONFIG = {
    "OHP": {
        "label_files": {
            "error_elbows": ("OHP/Labeled_Dataset/Labels/error_elbows.json",  "interval"),
            "error_knees":  ("OHP/Labeled_Dataset/Labels/error_knees.json",   "interval"),
        },
        "splits": {
            "train": "OHP/Labeled_Dataset/Splits/train_keys.json",
            "val":   "OHP/Labeled_Dataset/Splits/val_keys.json",
            "test":  "OHP/Labeled_Dataset/Splits/test_keys.json",
        },
        "key_format": "subject_rep",        # e.g. "66089_2"
    },
    "Squat": {
        "label_files": {
            "knees_inward":  ("Squat/Labeled_Dataset/Labels/error_knees_inward.json",  "interval"),
            "knees_forward": ("Squat/Labeled_Dataset/Labels/error_knees_forward.json", "interval"),
            "shallow_depth": ("Squat/Labeled_Dataset/Shallow_Squat_Error_Dataset/labels_shallow_depth.json", "frame_binary"),
        },
        "splits": {
            "train": "Squat/Labeled_Dataset/Splits/train_keys.json",
            "val":   "Squat/Labeled_Dataset/Splits/val_keys.json",
            "test":  "Squat/Labeled_Dataset/Splits/test_keys.json",
        },
        "key_format": "subject_rep",        # e.g. "46777_3"
    },
    "BarbellRow": {
        "label_files": {
            "lumbar_error": ("BarbellRow/Labeled_Dataset/Labels/labels_lumbar_error.json",      "binary"),
            "torso_angle":  ("BarbellRow/Labeled_Dataset/Labels/labels_torso_angle_error.json", "binary"),
        },
        "splits": {
            # Lumbar split used as primary (covers all labeled reps)
            "train": "BarbellRow/Labeled_Dataset/Splits/Splits_Lumbar_Error/train_ids.json",
            "val":   "BarbellRow/Labeled_Dataset/Splits/Splits_Lumbar_Error/val_ids.json",
            "test":  "BarbellRow/Labeled_Dataset/Splits/Splits_Lumbar_Error/test_ids.json",
        },
        "key_format": "subject_session_rep",  # e.g. "56067_3_51"
    },
}

# ── Data container ─────────────────────────────────────────────────────────────
@dataclass
class RepLabel:
    rep_key:    str
    subject_id: str
    exercise:   str
    errors:     List[str]           # names of error classes present
    intervals:  Dict[str, list]     # error_name → [[t_start, t_end], ...] or [] / {0,1}
    score:      float               # derived quality score ∈ [0, 1]
    multihot:   np.ndarray          # shape (n_error_classes,) binary float32

    def __repr__(self):
        return (f"RepLabel(key={self.rep_key}, score={self.score:.2f}, "
                f"errors={self.errors})")

# ── Quality score from temporal intervals ─────────────────────────────────────
def intervals_to_score(interval_dict: Dict[str, list], clip_duration_s: float = 8.0) -> float:
    """
    Score = 1 − clamp(total_error_seconds / clip_duration, 0, 1).
    Only processes 'interval' format values (lists of [t0, t1] segments).
    Binary values (0/1) are handled separately before calling this.
    """
    total_error = 0.0
    for val in interval_dict.values():
        if isinstance(val, list):                   # interval format
            for seg in val:
                if isinstance(seg, list) and len(seg) == 2:
                    total_error += max(0.0, seg[1] - seg[0])
        # binary int values contribute nothing here (handled via multihot)
    return float(np.clip(1.0 - total_error / clip_duration_s, 0.0, 1.0))

# ── Subject ID extraction ──────────────────────────────────────────────────────
def extract_subject_id(rep_key: str, key_format: str) -> str:
    """
    'subject_rep'         → "46777_3"    → subject = "46777"
    'subject_session_rep' → "56067_3_51" → subject = "56067"
    Subject is always the first underscore-delimited token.
    """
    return rep_key.split("_")[0]

# ── Load all labels for one exercise ──────────────────────────────────────────
def load_exercise_labels(exercise: str, dataset_root: Path = DATASET_ROOT) -> Dict[str, RepLabel]:
    ecfg        = EXERCISE_CONFIG[exercise]
    error_names = list(ecfg["label_files"].keys())
    key_format  = ecfg["key_format"]

    # Load each label file; normalise to {rep_key: value} (value = list or int)
    label_data:  Dict[str, Dict[str, object]] = {}
    all_keys = set()
    for ename, (rel_path, fmt) in ecfg["label_files"].items():
        d = json.load(open(dataset_root / rel_path))

        if fmt == "frame_binary":
            # ── Aggregate frame-level 3-part keys → rep-level 2-part keys ──
            # e.g. "37803_2_44" → rep "37803_2", if ANY frame=1 → rep=1
            from collections import defaultdict
            rep_frames = defaultdict(list)
            for k, v in d.items():
                parts = k.split("_")
                rep_key = f"{parts[0]}_{parts[1]}"  # subject_rep
                rep_frames[rep_key].append(v)
            d = {rep: 1 if any(v == 1 for v in frames) else 0
                 for rep, frames in rep_frames.items()}
            # Treat as binary from here on
        label_data[ename] = d
        all_keys.update(d.keys())

    results: Dict[str, RepLabel] = {}
    for key in sorted(all_keys):
        subject_id = extract_subject_id(key, key_format)

        # Build per-error info
        raw_vals = {ename: label_data[ename].get(key, 0 if EXERCISE_CONFIG[exercise]["label_files"][ename][1] in ("binary", "frame_binary") else [])
                    for ename in error_names}

        # Binary/frame_binary: 0/1 int  |  Interval: list (possibly empty)
        def _is_error(val, fmt):
            if fmt in ("binary", "frame_binary"):
                return bool(val)
            return bool(val)            # non-empty list → True

        errors_present = [
            ename for ename in error_names
            if _is_error(raw_vals[ename], ecfg["label_files"][ename][1])
        ]
        multihot = np.array(
            [1.0 if ename in errors_present else 0.0 for ename in error_names],
            dtype=np.float32,
        )

        # Quality score: interval → duration-based; binary → 0 or 1
        has_any_interval = any(
            ecfg["label_files"][e][1] == "interval" for e in error_names
        )
        if has_any_interval:
            # Pass only interval-type values to intervals_to_score
            interval_vals = {
                e: raw_vals[e]
                for e in error_names
                if ecfg["label_files"][e][1] == "interval"
            }
            score = intervals_to_score(interval_vals)
            # Also penalise binary/frame_binary errors
            n_binary_errors = sum(
                1 for e in error_names
                if ecfg["label_files"][e][1] in ("binary", "frame_binary") and raw_vals[e]
            )
            n_binary_total = sum(
                1 for e in error_names
                if ecfg["label_files"][e][1] in ("binary", "frame_binary")
            )
            if n_binary_total > 0:
                score = score * (1.0 - 0.5 * n_binary_errors / n_binary_total)
        else:
            # All-binary exercise: score = fraction of error-free labels
            score = 1.0 - multihot.mean()

        results[key] = RepLabel(
            rep_key    = key,
            subject_id = subject_id,
            exercise   = exercise,
            errors     = errors_present,
            intervals  = raw_vals,
            score      = score,
            multihot   = multihot,
        )
    return results

# ── Load splits ────────────────────────────────────────────────────────────────
def load_splits(exercise: str, dataset_root: Path = DATASET_ROOT) -> Dict[str, List[str]]:
    ecfg = EXERCISE_CONFIG[exercise]
    return {
        split: json.load(open(dataset_root / rel_path))
        for split, rel_path in ecfg["splits"].items()
    }

# ── Load everything ────────────────────────────────────────────────────────────
ALL_LABELS: Dict[str, Dict[str, RepLabel]] = {}
ALL_SPLITS: Dict[str, Dict[str, List[str]]] = {}

for ex in EXERCISE_CONFIG:
    ALL_LABELS[ex] = load_exercise_labels(ex)
    ALL_SPLITS[ex] = load_splits(ex)
    splits = ALL_SPLITS[ex]
    error_names = list(EXERCISE_CONFIG[ex]["label_files"].keys())
    fmt_str = ", ".join(
        f"{e}({'B' if EXERCISE_CONFIG[ex]['label_files'][e][1]=='binary' else 'I'})"
        for e in error_names
    )
    print(f"{ex:12s}  labels={len(ALL_LABELS[ex]):6d}  "
          f"train={len(splits['train']):5d}  val={len(splits['val']):4d}  test={len(splits['test']):4d}  "
          f"  [{fmt_str}]  (B=binary, I=interval)")

print("\nSample RepLabels:")
for ex in EXERCISE_CONFIG:
    sample = next(iter(ALL_LABELS[ex].values()))
    print(f"  {ex}: {sample}")


In [ ]:
"""
Class imbalance analysis + quality score distribution for all exercises.
"""
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Fitness-AQA Dataset Analysis", fontsize=14, fontweight="bold")

for col_idx, exercise in enumerate(["OHP", "Squat", "BarbellRow"]):
    labels_dict  = ALL_LABELS[exercise]
    error_names  = list(EXERCISE_CONFIG[exercise]["label_files"].keys())
    all_labels   = list(labels_dict.values())

    # ── Top row: class imbalance bar chart ─────────────────────────────────
    ax_bar = axes[0, col_idx]
    counts_pos = [sum(1 for r in all_labels if r.multihot[i] == 1) for i in range(len(error_names))]
    counts_neg = [len(all_labels) - c for c in counts_pos]
    x = np.arange(len(error_names))
    b1 = ax_bar.bar(x - 0.2, counts_neg, 0.4, label="No error", color="steelblue", alpha=0.8)
    b2 = ax_bar.bar(x + 0.2, counts_pos, 0.4, label="Error",    color="tomato",    alpha=0.8)
    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(error_names, rotation=20, ha="right", fontsize=8)
    ax_bar.set_title(f"{exercise} — Class Balance")
    ax_bar.set_ylabel("# reps")
    ax_bar.legend(fontsize=8)
    # annotate percentages
    for xi, cp in zip(x, counts_pos):
        pct = 100 * cp / len(all_labels)
        ax_bar.text(xi + 0.2, cp + 5, f"{pct:.0f}%", ha="center", fontsize=7, color="darkred")

    # ── Bottom row: quality score histogram ────────────────────────────────
    ax_hist = axes[1, col_idx]
    scores = [r.score for r in all_labels]
    ax_hist.hist(scores, bins=20, color="mediumseagreen", edgecolor="white", alpha=0.85)
    ax_hist.axvline(np.mean(scores), color="red",    linestyle="--", linewidth=1.5, label=f"Mean={np.mean(scores):.2f}")
    ax_hist.axvline(np.median(scores), color="navy", linestyle=":",  linewidth=1.5, label=f"Median={np.median(scores):.2f}")
    ax_hist.set_title(f"{exercise} — Quality Score Distribution")
    ax_hist.set_xlabel("Score (0=all errors, 1=clean)")
    ax_hist.set_ylabel("# reps")
    ax_hist.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../docs/assets/dataset_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → dataset_analysis.png")

# ── Subject-level stats (data leakage check) ──────────────────────────────────
print("\n── Subject counts per exercise ──")
for exercise in ["OHP", "Squat", "BarbellRow"]:
    subjects = {r.subject_id for r in ALL_LABELS[exercise].values()}
    reps_per_subj = {}
    for r in ALL_LABELS[exercise].values():
        reps_per_subj.setdefault(r.subject_id, 0)
        reps_per_subj[r.subject_id] += 1
    counts = list(reps_per_subj.values())
    print(f"  {exercise:12s}: {len(subjects):4d} subjects | "
          f"reps/subject: min={min(counts)}, max={max(counts)}, mean={np.mean(counts):.1f}")


---
## Stage 4 — Pose Extraction (MediaPipe)

**Input:** RGB video clips at `VIDEO_ROOT/{exercise}/{rep_key}.mp4`  
**Output:** NumPy array `(T, 33, 3)` saved to `POSE_ROOT/{exercise}/{rep_key}.npy`

MediaPipe Pose detects 33 body landmarks per frame: coordinates `(x, y, z)` in normalised image space.  
`z` is a pseudo-depth relative to the hip centre.

> **Install:** `pip install mediapipe opencv-python`

### Data Availability Check

Run this cell to see which pose/processed data is already available.  
If all data exists, you can skip extraction and processing cells.

In [ ]:
"""
Check availability of pre-extracted poses and processed data.
"""
import os
from pathlib import Path

def check_data_availability():
    """Check how much pose and processed data is already available."""
    results = {}
    
    for exercise in ["OHP", "Squat", "BarbellRow"]:
        # Count available poses
        pose_dir = POSE_ROOT / exercise
        poses_available = len(list(pose_dir.glob("*.npy"))) if pose_dir.exists() else 0
        
        # Count processed data
        processed_dir = PROCESSED_ROOT / exercise 
        processed_available = len(list(processed_dir.glob("*.npy"))) if processed_dir.exists() else 0
        
        # Total labeled samples for this exercise
        total_labeled = len(ALL_LABELS[exercise])
        
        results[exercise] = {
            'poses': poses_available,
            'processed': processed_available, 
            'total_labeled': total_labeled
        }
        
        # Status indicators
        pose_status = "✅ Complete" if poses_available >= total_labeled * 0.95 else f"⚠️  Partial ({poses_available}/{total_labeled})"
        processed_status = "✅ Complete" if processed_available >= total_labeled * 0.95 else f"⚠️  Partial ({processed_available}/{total_labeled})"
        
        print(f"\n{exercise}:")
        print(f"  📦 Raw poses:      {poses_available:4d}/{total_labeled}  {pose_status}")
        print(f"  🔄 Processed data: {processed_available:4d}/{total_labeled}  {processed_status}")
    
    return results

# Run the check
print("🔍 Checking data availability...")
data_status = check_data_availability()

# Provide recommendations
print("\n" + "="*60)
print("📋 RECOMMENDATIONS:")

all_poses_complete = all(status['poses'] >= status['total_labeled'] * 0.95 for status in data_status.values())
all_processed_complete = all(status['processed'] >= status['total_labeled'] * 0.95 for status in data_status.values())

if all_poses_complete and all_processed_complete:
    print("🎉 ALL DATA AVAILABLE! You can skip to Stage 6 (model training)")
elif all_poses_complete:
    print("📦 Pose extraction complete. You can skip pose extraction cells.")
    print("🔄 Run processing cells to generate processed data.")
else:
    print("⚡ Missing pose data. Run pose extraction cells first.")
    
print("\n💡 TIP: If data is complete, you can:")
print("   - Skip cells that extract poses (if poses are complete)")
print("   - Skip cells that process poses (if processed data is complete)")  
print("   - Jump directly to model training and evaluation")

In [ ]:
"""
PoseExtractor — MediaPipe Pose Landmarker Tasks API (mediapipe ≥ 0.10).
Outputs (T, 33, 3) numpy array per video: normalised [x, y, z] per landmark.
"""
import numpy as np
import urllib.request
from pathlib import Path
from typing import Optional, Dict

try:
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python as _mp_python
    from mediapipe.tasks.python import vision as _mp_vision
    MP_AVAILABLE = True
except ImportError:
    MP_AVAILABLE = False
    print("⚠  mediapipe/opencv not installed.  Run:  pip install mediapipe opencv-python")

# ── Model download ─────────────────────────────────────────────────────────────
MODEL_URL  = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
              "pose_landmarker_lite/float16/1/pose_landmarker_lite.task")
MODEL_PATH = Path.home() / ".mediapipe" / "pose_landmarker_lite.task"

def _ensure_model(model_path: Path = MODEL_PATH) -> Path:
    if not model_path.exists():
        model_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"  Downloading pose landmarker model → {model_path} …", end=" ", flush=True)
        urllib.request.urlretrieve(MODEL_URL, model_path)
        print("done.")
    return model_path


def _build_landmarker(model_path: Path, use_gpu: bool):
    """Build MediaPipe PoseLandmarker with requested delegate."""
    base_kwargs = {"model_asset_path": str(model_path)}
    if use_gpu and hasattr(_mp_python.BaseOptions, "Delegate"):
        base_kwargs["delegate"] = _mp_python.BaseOptions.Delegate.GPU

    options = _mp_vision.PoseLandmarkerOptions(
        base_options=_mp_python.BaseOptions(**base_kwargs),
        running_mode=_mp_vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return _mp_vision.PoseLandmarker.create_from_options(options)


# ── PoseExtractor ──────────────────────────────────────────────────────────────
class PoseExtractor:
    """
    Extracts MediaPipe Pose Landmarker landmarks from a video.
    Uses the Tasks API (mediapipe ≥ 0.10).

    Output: np.ndarray shape (T, 33, 3) — NaN rows for undetected frames.
    Coordinates: normalised image-space (x, y) + pseudo-depth z relative to hips.
    """

    N_LANDMARKS = 33

    LANDMARK_NAMES = [
        "nose", "left_eye_inner", "left_eye", "left_eye_outer",
        "right_eye_inner", "right_eye", "right_eye_outer",
        "left_ear", "right_ear", "mouth_left", "mouth_right",
        "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
        "left_wrist", "right_wrist", "left_pinky", "right_pinky",
        "left_index", "right_index", "left_thumb", "right_thumb",
        "left_hip", "right_hip", "left_knee", "right_knee",
        "left_ankle", "right_ankle", "left_heel", "right_heel",
        "left_foot_index", "right_foot_index",
    ]

    def __init__(self, model_path: Path = MODEL_PATH, prefer_gpu: bool = True):
        self.backend = "unavailable"
        if not MP_AVAILABLE:
            self._landmarker = None
            return

        _ensure_model(model_path)

        self._landmarker = None
        if prefer_gpu:
            try:
                self._landmarker = _build_landmarker(model_path, use_gpu=True)
                self.backend = "gpu"
            except Exception as e:
                print(f"⚠  GPU delegate unavailable ({e}); falling back to CPU.")

        if self._landmarker is None:
            self._landmarker = _build_landmarker(model_path, use_gpu=False)
            self.backend = "cpu"

    def extract(self, video_path: Path) -> Optional[np.ndarray]:
        """
        Returns (T, 33, 3) float32 or None if video is unreadable.
        """
        if self._landmarker is None:
            return None

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"  ERROR: cannot open {video_path}")
            return None

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb         = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img      = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result      = self._landmarker.detect(mp_img)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                lm = result.pose_landmarks[0]        # first (and only) person
                frame_arr = np.array(
                    [[l.x, l.y, l.z] for l in lm], dtype=np.float32
                )                                    # (33, 3)
            else:
                frame_arr = np.full((self.N_LANDMARKS, 3), np.nan, dtype=np.float32)
            frames.append(frame_arr)

        cap.release()
        if not frames:
            return None
        return np.stack(frames, axis=0)              # (T, 33, 3)

    def close(self):
        if self._landmarker is not None:
            self._landmarker.close()

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()


# ── Batch extraction ───────────────────────────────────────────────────────────
def extract_all_poses(
    exercise:    str,
    pose_root:   Path = POSE_ROOT,
    overwrite:   bool = False,
    max_videos:  Optional[int] = None,
    prefer_gpu:  bool = True,
) -> Dict[str, Path]:
    """
    Extract poses for every labeled rep in `exercise`.

    Video location : DATASET_ROOT/{exercise}/Labeled_Dataset/videos/{rep_key}.mp4
    Pose output    : {pose_root}/{exercise}/{rep_key}.npy
    Returns        : dict rep_key → saved .npy path
    """
    if not MP_AVAILABLE:
        print("MediaPipe not available.")
        return {}

    video_dir = video_dir_for_exercise(exercise)
    out_dir   = pose_root / exercise
    out_dir.mkdir(parents=True, exist_ok=True)

    keys = list(ALL_LABELS[exercise].keys())
    if max_videos:
        keys = keys[:max_videos]

    saved, missing = {}, []

    with PoseExtractor(prefer_gpu=prefer_gpu) as extractor:
        print(f"[{exercise}] MediaPipe backend: {extractor.backend}")
        for i, rep_key in enumerate(keys):
            out_path = out_dir / f"{rep_key}.npy"
            if out_path.exists() and not overwrite:
                saved[rep_key] = out_path
                continue

            vid_path = video_dir / f"{rep_key}.mp4"
            if not vid_path.exists():
                missing.append(rep_key)
                continue

            landmarks = extractor.extract(vid_path)
            if landmarks is not None:
                np.save(out_path, landmarks)
                saved[rep_key] = out_path

            if (i + 1) % 50 == 0:
                print(f"  [{exercise}] {i+1}/{len(keys)} processed …")

    print(f"[{exercise}] Saved: {len(saved)} | Missing videos: {len(missing)}")
    return saved


# ── Smoke test on first available video ───────────────────────────────────────
print("PoseExtractor ready.")
print("To extract all OHP poses:  extract_all_poses('OHP', prefer_gpu=True)")
print()

if not MP_AVAILABLE:
    print("MediaPipe not installed. Run:  pip install mediapipe opencv-python")
else:
    test_vid = next(DATASET_ROOT.rglob("*.mp4"), None)
    if test_vid:
        print(f"Smoke-testing on: {test_vid.relative_to(DATASET_ROOT)}")
        with PoseExtractor(prefer_gpu=True) as pe:
            print(f"Backend selected: {pe.backend}")
            arr = pe.extract(test_vid)
        if arr is not None:
            nan_frames = int(np.isnan(arr).any(axis=(1, 2)).sum())
            print(f"✓  Extraction OK")
            print(f"   Shape          : {arr.shape}  (T={arr.shape[0]} frames)")
            print(f"   Undetected     : {nan_frames} / {arr.shape[0]} frames")
            print(f"   Coord range    : x=[{np.nanmin(arr[:,:,0]):.2f}, {np.nanmax(arr[:,:,0]):.2f}]  "
                  f"y=[{np.nanmin(arr[:,:,1]):.2f}, {np.nanmax(arr[:,:,1]):.2f}]")
        else:
            print("⚠  Extraction returned None — check video file.")
    else:
        print("No .mp4 files found under DATASET_ROOT.")


In [ ]:
"""
🎯 POSE EXTRACTION - OHP & Squat

⚠️  SKIP THIS CELL if poses are already available (check output of data availability checker above)

This cell extracts poses from video files for OHP and Squat exercises.
Only runs extraction if poses are missing or if you want to overwrite existing data.
"""

# Check if poses already exist
ohp_pose_dir = POSE_ROOT / "OHP"
squat_pose_dir = POSE_ROOT / "Squat"

ohp_existing = len(list(ohp_pose_dir.glob("*.npy"))) if ohp_pose_dir.exists() else 0
squat_existing = len(list(squat_pose_dir.glob("*.npy"))) if squat_pose_dir.exists() else 0

ohp_total = len(ALL_LABELS["OHP"]) 
squat_total = len(ALL_LABELS["Squat"])

print(f"📊 Current pose availability:")
print(f"   OHP: {ohp_existing}/{ohp_total} poses available")
print(f"   Squat: {squat_existing}/{squat_total} poses available")

# Ask user for confirmation if data exists
run_extraction = True
if ohp_existing > 0 or squat_existing > 0:
    print("\n ⚠️  EXISTING POSE DATA DETECTED!")
    print("    Set overwrite=True below to reprocess existing poses")
    print("    Set skip_extraction=True below to skip this step entirely")
    
# Configuration
skip_extraction = False  # Set to True to skip extraction entirely
overwrite_existing = False  # Set to True to overwrite existing poses

if skip_extraction:
    print("\n⏭️  SKIPPING pose extraction (skip_extraction=True)")
else:
    print(f"\n🚀 Starting pose extraction (overwrite={overwrite_existing})")
    
    # Extract OHP poses
    if ohp_existing < ohp_total or overwrite_existing:
        print("\n📹 Extracting OHP poses...")
        extract_all_poses("OHP", max_videos=None, prefer_gpu=True, overwrite=overwrite_existing)
    else:
        print(f"✅ OHP poses already complete ({ohp_existing}/{ohp_total})")
    
    # Extract Squat poses  
    if squat_existing < squat_total or overwrite_existing:
        print("\n📹 Extracting Squat poses...")
        extract_all_poses("Squat", max_videos=None, prefer_gpu=True, overwrite=overwrite_existing)
    else:
        print(f"✅ Squat poses already complete ({squat_existing}/{squat_total})")
        
    print("\n✨ Pose extraction complete!")

print(f"\n📁 Pose data location: {POSE_ROOT}")
print("💡 TIP: Extracted poses are saved as .npy files with shape (T, 33, 3)")
print("       where T=frames, 33=landmarks, 3=coordinates (x,y,z)")

In [ ]:
def extract_barbell_poses_from_images(
    pose_root:  Path = POSE_ROOT,
    overwrite:  bool = False,
    prefer_gpu: bool = True,
) -> Dict[str, Path]:
    """
    BarbellRow: group {subject}_{session}_{frame}.jpg by rep key {subject}_{session},
    sort frames numerically, stack into (T, 33, 3) — same shape as video extraction.
    """
    if not MP_AVAILABLE:
        print("MediaPipe not available.")
        return {}

    out_dir = pose_root / "BarbellRow"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Build rep_key → sorted list of jpg paths
    rep_to_frames: Dict[str, List] = defaultdict(list)
    for jpg in BARBELL_IMAGES_DIR.glob("*.jpg"):
        parts = jpg.stem.split("_")
        if len(parts) >= 3:
            rep_key   = f"{parts[0]}_{parts[1]}_{parts[2]}" # "52701_11_14" ← matches label key
            frame_num = int(parts[3]) if len(parts) > 3 else int(parts[2])
            rep_to_frames[rep_key].append((frame_num, jpg))

    for k in rep_to_frames:
        rep_to_frames[k] = [p for _, p in sorted(rep_to_frames[k])]

    labeled_reps = set(ALL_LABELS["BarbellRow"].keys())
    keys_to_process = sorted(labeled_reps & set(rep_to_frames.keys()))

    saved, missing, failed = {}, [], []

    with PoseExtractor(prefer_gpu=prefer_gpu) as extractor:
        print(f"[BarbellRow] MediaPipe backend: {extractor.backend}")
        for i, rep_key in enumerate(keys_to_process):
            out_path = out_dir / f"{rep_key}.npy"
            if out_path.exists() and not overwrite:
                saved[rep_key] = out_path
                continue

            frame_paths = rep_to_frames[rep_key]
            frame_landmarks = []

            for fp in frame_paths:
                frame = cv2.imread(str(fp))
                if frame is None:
                    frame_landmarks.append(np.full((33, 3), np.nan, dtype=np.float32))
                    continue

                rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                result = extractor._landmarker.detect(mp_img)

                if result.pose_landmarks and len(result.pose_landmarks) > 0:
                    lm        = result.pose_landmarks[0]
                    frame_arr = np.array([[l.x, l.y, l.z] for l in lm], dtype=np.float32)
                else:
                    frame_arr = np.full((33, 3), np.nan, dtype=np.float32)

                frame_landmarks.append(frame_arr)

            if not frame_landmarks:
                failed.append(rep_key)
                continue

            landmarks = np.stack(frame_landmarks, axis=0)  # (T, 33, 3)
            np.save(out_path, landmarks)
            saved[rep_key] = out_path

            if (i + 1) % 200 == 0:
                print(f"  [BarbellRow] {i+1}/{len(keys_to_process)} reps processed …")

    print(f"[BarbellRow] Saved: {len(saved)} | No label match: {len(missing)} | Failed: {len(failed)}")
    return saved


# Run the verification cell first, then:
# extract_barbell_poses_from_images(prefer_gpu=True)

In [ ]:
"""
🎯 POSE EXTRACTION - BarbellRow

⚠️  SKIP THIS CELL if BarbellRow poses are already available

This cell extracts poses from image frames for BarbellRow exercise.
Only runs extraction if poses are missing or if you want to overwrite existing data.
"""

# Check if BarbellRow poses already exist
barbell_pose_dir = POSE_ROOT / "BarbellRow"
barbell_existing = len(list(barbell_pose_dir.glob("*.npy"))) if barbell_pose_dir.exists() else 0
barbell_total = len(ALL_LABELS["BarbellRow"])

print(f"📊 BarbellRow pose availability: {barbell_existing}/{barbell_total}")

# Configuration  
skip_barbell_extraction = False  # Set to True to skip extraction entirely
overwrite_barbell = False       # Set to True to overwrite existing poses

if skip_barbell_extraction:
    print("\n⏭️  SKIPPING BarbellRow pose extraction (skip_barbell_extraction=True)")
elif barbell_existing >= barbell_total and not overwrite_barbell:
    print(f"\n✅ BarbellRow poses already complete ({barbell_existing}/{barbell_total})")
    print("    Set overwrite_barbell=True above to reprocess")
else:
    print(f"\n🚀 Starting BarbellRow pose extraction (overwrite={overwrite_barbell})")
    extract_barbell_poses_from_images(prefer_gpu=True, overwrite=overwrite_barbell)
    print("\n✨ BarbellRow pose extraction complete!")

print(f"\n📁 BarbellRow poses location: {barbell_pose_dir}")
print("💡 TIP: Each rep is saved as {subject}_{session}_{rep}.npy")

### Alternative: TorchVision Keypoint R-CNN (GPU)

If MediaPipe GPU delegate is unavailable, this uses **TorchVision Keypoint R-CNN** on CUDA.
Converts COCO-17 keypoints to the `(T, 33, 3)` landmark format. Unmapped landmarks are left as NaN and handled by `fill_nan_frames()`.

In [ ]:
"""
PyTorch Keypoint R-CNN based pose extraction fallback.
Outputs (T, 33, 3) to stay compatible with existing pipeline.
"""
import numpy as np
import torch
import torchvision
from torchvision.transforms import functional as TF
from pathlib import Path
from typing import Optional, Dict
import cv2

# COCO keypoints -> index in our 33-landmark schema (MediaPipe-like slots)
COCO_TO_33 = {
    0: 0,    # nose
    1: 2,    # left_eye -> left_eye
    2: 5,    # right_eye -> right_eye
    3: 7,    # left_ear
    4: 8,    # right_ear
    5: 11,   # left_shoulder
    6: 12,   # right_shoulder
    7: 13,   # left_elbow
    8: 14,   # right_elbow
    9: 15,   # left_wrist
    10: 16,  # right_wrist
    11: 23,  # left_hip
    12: 24,  # right_hip
    13: 25,  # left_knee
    14: 26,  # right_knee
    15: 27,  # left_ankle
    16: 28,  # right_ankle
}

class TorchPoseExtractor:
    """
    Extract pose keypoints frame-by-frame using Keypoint R-CNN.
    Returns landmarks in shape (T, 33, 3): x,y normalized to [0,1], z=0.
    """

    N_LANDMARKS = 33

    def __init__(self, score_thr: float = 0.5, kpt_thr: float = 0.2, device: str = "auto"):
        if device == "auto":
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)

        weights = torchvision.models.detection.KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
        self.model = torchvision.models.detection.keypointrcnn_resnet50_fpn(weights=weights)
        self.model.to(self.device).eval()
        self.score_thr = score_thr
        self.kpt_thr = kpt_thr
        self.backend = f"torchvision:{self.device.type}"

    @torch.no_grad()
    def _extract_frame(self, frame_bgr: np.ndarray) -> np.ndarray:
        h, w = frame_bgr.shape[:2]
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        x = TF.to_tensor(frame_rgb).to(self.device)
        out = self.model([x])[0]

        lm33 = np.full((self.N_LANDMARKS, 3), np.nan, dtype=np.float32)
        if len(out["scores"]) == 0:
            return lm33

        best_idx = int(torch.argmax(out["scores"]).item())
        if float(out["scores"][best_idx].item()) < self.score_thr:
            return lm33

        kpts = out["keypoints"][best_idx].detach().cpu().numpy()      # (17,3) x,y,v
        kpt_scores = out["keypoints_scores"][best_idx].detach().cpu().numpy() if "keypoints_scores" in out else np.ones(17)

        for coco_idx, tgt_idx in COCO_TO_33.items():
            if kpt_scores[coco_idx] >= self.kpt_thr:
                x_px, y_px = kpts[coco_idx, 0], kpts[coco_idx, 1]
                lm33[tgt_idx, 0] = np.clip(x_px / max(w, 1), 0.0, 1.0)
                lm33[tgt_idx, 1] = np.clip(y_px / max(h, 1), 0.0, 1.0)
                lm33[tgt_idx, 2] = 0.0

        return lm33

    def extract(self, video_path: Path) -> Optional[np.ndarray]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"  ERROR: cannot open {video_path}")
            return None

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(self._extract_frame(frame))

        cap.release()
        if not frames:
            return None
        return np.stack(frames, axis=0).astype(np.float32)


def extract_all_poses_torch(
    exercise: str,
    pose_root: Path = POSE_ROOT,
    overwrite: bool = False,
    max_videos: Optional[int] = None,
    device: str = "auto",
) -> Dict[str, Path]:
    """
    GPU-friendly alternative pose extraction using TorchVision keypoint model.
    Saves outputs to same location/shape as MediaPipe extractor.
    """
    video_dir = video_dir_for_exercise(exercise)
    out_dir = pose_root / exercise
    out_dir.mkdir(parents=True, exist_ok=True)

    keys = list(ALL_LABELS[exercise].keys())
    if max_videos:
        keys = keys[:max_videos]

    saved, missing = {}, []
    extractor = TorchPoseExtractor(device=device)
    print(f"[{exercise}] Torch pose backend: {extractor.backend}")

    for i, rep_key in enumerate(keys):
        out_path = out_dir / f"{rep_key}.npy"
        if out_path.exists() and not overwrite:
            saved[rep_key] = out_path
            continue

        vid_path = video_dir / f"{rep_key}.mp4"
        if not vid_path.exists():
            missing.append(rep_key)
            continue

        landmarks = extractor.extract(vid_path)
        if landmarks is not None:
            np.save(out_path, landmarks)
            saved[rep_key] = out_path

        if (i + 1) % 50 == 0:
            print(f"  [{exercise}] {i+1}/{len(keys)} processed …")

    print(f"[{exercise}] Saved: {len(saved)} | Missing videos: {len(missing)}")
    return saved


print("Torch pose extractor ready.")
print("Example: extract_all_poses_torch('OHP', max_videos=100)")

In [ ]:
def extract_barbell_poses_from_images_torch(
    pose_root: Path = POSE_ROOT,
    overwrite: bool = False,
    device: str = "auto",
) -> Dict[str, Path]:
    """
    BarbellRow extraction for image-based data using TorchPoseExtractor.


    IMPORTANT: Label keys for BarbellRow match image stems exactly:
      {subject}_{rep}_{frame}.jpg
    so we save one landmark array per labeled frame with shape (1, 33, 3).
    """
    out_dir = pose_root / "BarbellRow"
    out_dir.mkdir(parents=True, exist_ok=True)

    extractor = TorchPoseExtractor(device=device)
    print(f"[BarbellRow] Torch pose backend: {extractor.backend}")

    labeled_keys = set(ALL_LABELS["BarbellRow"].keys())
    jpg_map = {p.stem: p for p in BARBELL_IMAGES_DIR.glob("*.jpg")}
    keys_to_process = sorted(labeled_keys & set(jpg_map.keys()))

    saved, missing, failed = {}, [], []

    for i, rep_key in enumerate(keys_to_process):
        out_path = out_dir / f"{rep_key}.npy"
        if out_path.exists() and not overwrite:
            saved[rep_key] = out_path
            continue

        img_path = jpg_map.get(rep_key)
        if img_path is None:
            missing.append(rep_key)
            continue

        frame = cv2.imread(str(img_path))
        if frame is None:
            failed.append(rep_key)
            continue

        lm = extractor._extract_frame(frame)            # (33,3)
        lm_seq = lm[np.newaxis, ...].astype(np.float32)  # (1,33,3)
        np.save(out_path, lm_seq)
        saved[rep_key] = out_path

        if (i + 1) % 500 == 0:
            print(f"  [BarbellRow] {i+1}/{len(keys_to_process)} frames processed …")

    print(f"[BarbellRow] Saved: {len(saved)} | Missing: {len(missing)} | Failed: {len(failed)}")
    return saved


print("Torch Barbell image extractor ready.")
print("Example: extract_barbell_poses_from_images_torch(device='auto')")

---
## Stage 5 — Pose Normalization

**Goals:**
1. **Translation invariance** — subtract hip midpoint from all landmarks each frame
2. **Scale invariance** — divide by shoulder-to-shoulder distance
3. **Optional feature augmentation:**
   - **Joint angles** (12 angles: elbows, knees, hips, shoulders, ankles, wrists)
   - **Velocities** — first temporal derivative of normalised landmarks

MediaPipe landmark indices used:  
`left_hip=23, right_hip=24, left_shoulder=11, right_shoulder=12`

In [ ]:
"""
Pose normalization: center + scale, compute joint angles and velocities.
"""
import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path

# ── MediaPipe landmark index constants ────────────────────────────────────────
IDX = {
    "nose": 0,
    "left_shoulder": 11, "right_shoulder": 12,
    "left_elbow":    13, "right_elbow":    14,
    "left_wrist":    15, "right_wrist":    16,
    "left_hip":      23, "right_hip":      24,
    "left_knee":     25, "right_knee":     26,
    "left_ankle":    27, "right_ankle":    28,
}

# ── Joint angle triplets: (joint, proximal, distal) ──────────────────────────
ANGLE_TRIPLETS = [
    ("left_elbow",    "left_shoulder",  "left_wrist"),
    ("right_elbow",   "right_shoulder", "right_wrist"),
    ("left_shoulder", "left_hip",       "left_elbow"),
    ("right_shoulder","right_hip",      "right_elbow"),
    ("left_hip",      "left_shoulder",  "left_knee"),
    ("right_hip",     "right_shoulder", "right_knee"),
    ("left_knee",     "left_hip",       "left_ankle"),
    ("right_knee",    "right_hip",      "right_ankle"),
    ("left_ankle",    "left_knee",      "left_hip"),
    ("right_ankle",   "right_knee",     "right_hip"),
    ("left_wrist",    "left_elbow",     "left_shoulder"),
    ("right_wrist",   "right_elbow",    "right_shoulder"),
]  # 12 angles


def _angle_between(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """Angle at vertex b, given points a–b–c. Returns degrees."""
    v1 = a - b
    v2 = c - b
    cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0))))


def normalize_pose(landmarks: np.ndarray) -> np.ndarray:
    """
    Center and scale a pose sequence.

    Parameters
    ----------
    landmarks : (T, 33, 3) float32

    Returns
    -------
    normalised : (T, 33, 3) float32
    """
    T = landmarks.shape[0]
    normed = landmarks.copy()

    left_hip  = landmarks[:, IDX["left_hip"], :]    # (T, 3)
    right_hip = landmarks[:, IDX["right_hip"], :]   # (T, 3)
    hip_mid   = (left_hip + right_hip) / 2.0        # (T, 3)

    # 1. Translation: subtract hip midpoint
    normed -= hip_mid[:, np.newaxis, :]             # broadcast over 33 landmarks

    # 2. Scale: shoulder distance
    left_sh   = normed[:, IDX["left_shoulder"],  :]
    right_sh  = normed[:, IDX["right_shoulder"], :]
    scale     = np.linalg.norm(left_sh - right_sh, axis=1, keepdims=True)  # (T, 1)
    scale     = np.where(scale < 1e-6, 1.0, scale)                         # avoid div-by-zero
    normed   /= scale[:, np.newaxis, :]             # (T, 33, 3)

    return normed.astype(np.float32)


def compute_angles(landmarks: np.ndarray) -> np.ndarray:
    """
    Compute 12 joint angles per frame.

    Parameters
    ----------
    landmarks : (T, 33, 3) — already normalised or raw

    Returns
    -------
    angles : (T, 12) degrees, float32
    """
    T = landmarks.shape[0]
    angles = np.zeros((T, len(ANGLE_TRIPLETS)), dtype=np.float32)
    for t in range(T):
        for j, (joint, prox, dist) in enumerate(ANGLE_TRIPLETS):
            a = landmarks[t, IDX[prox], :]
            b = landmarks[t, IDX[joint], :]
            c = landmarks[t, IDX[dist], :]
            if np.any(np.isnan(a)) or np.any(np.isnan(b)) or np.any(np.isnan(c)):
                angles[t, j] = 0.0
            else:
                angles[t, j] = _angle_between(a, b, c)
    return angles


def compute_velocity(features: np.ndarray) -> np.ndarray:
    """
    First-order temporal derivative.  Frame 0 = zeros.

    Parameters
    ----------
    features : (T, F)
    Returns  : (T, F) velocity
    """
    vel = np.zeros_like(features)
    vel[1:] = features[1:] - features[:-1]
    return vel.astype(np.float32)


def fill_nan_frames(landmarks: np.ndarray) -> np.ndarray:
    """
    Linear interpolation for frames where MediaPipe failed (NaN landmarks).
    Falls back to nearest valid frame if all are NaN.
    """
    T = landmarks.shape[0]
    flat = landmarks.reshape(T, -1)   # (T, 99)
    nan_rows = np.any(np.isnan(flat), axis=1)

    if nan_rows.all():
        return np.zeros_like(landmarks)
    if not nan_rows.any():
        return landmarks

    valid_idx = np.where(~nan_rows)[0]
    for col in range(flat.shape[1]):
        nan_mask = np.isnan(flat[:, col])
        if nan_mask.any():
            flat[:, col] = np.interp(np.arange(T), valid_idx, flat[valid_idx, col])

    return flat.reshape(landmarks.shape).astype(np.float32)


def build_feature_vector(
    landmarks: np.ndarray,
    use_angles: bool   = True,
    use_velocity: bool = True,
) -> np.ndarray:
    """
    Full feature pipeline: normalise → flatten → append angles → append velocity.

    Parameters
    ----------
    landmarks    : (T, 33, 3)

    Returns
    -------
    features : (T, F)  where F depends on use_angles / use_velocity flags
    """
    lm = fill_nan_frames(landmarks)
    lm = normalize_pose(lm)                         # (T, 33, 3)

    feat = lm.reshape(lm.shape[0], -1)             # (T, 99)

    if use_angles:
        angles = compute_angles(lm)                 # (T, 12)
        feat = np.concatenate([feat, angles], axis=1)  # (T, 111)

    if use_velocity:
        vel = compute_velocity(feat)                # (T, F)
        feat = np.concatenate([feat, vel], axis=1) # (T, 2F)

    return feat.astype(np.float32)


# ── Batch normalise + save ─────────────────────────────────────────────────────
def process_all_poses(
    exercise: str,
    pose_root: Path      = POSE_ROOT,
    processed_root: Path = PROCESSED_ROOT,
    use_angles: bool     = cfg.use_angles,
    use_velocity: bool   = cfg.use_velocity,
    overwrite: bool      = False,
) -> Dict[str, Path]:
    """
    Load raw .npy poses → normalise → save processed .npy.
    Returns dict: rep_key → processed .npy path.
    """
    in_dir  = pose_root      / exercise
    out_dir = processed_root / exercise
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = {}
    for npy_path in sorted(in_dir.glob("*.npy")):
        rep_key  = npy_path.stem
        out_path = out_dir / npy_path.name
        if out_path.exists() and not overwrite:
            saved[rep_key] = out_path
            continue
        landmarks = np.load(npy_path)                       # (T, 33, 3)
        features  = build_feature_vector(landmarks, use_angles, use_velocity)
        np.save(out_path, features)
        saved[rep_key] = out_path

    print(f"[{exercise}] Processed: {len(saved)} files → {out_dir}")
    return saved


# ── Sanity test with synthetic data ───────────────────────────────────────────
T_test  = 73  # arbitrary clip length
fake_lm = np.random.randn(T_test, 33, 3).astype(np.float32)
feat    = build_feature_vector(fake_lm, use_angles=cfg.use_angles, use_velocity=cfg.use_velocity)
print(f"Synthetic test — input: (T=73, 33, 3)  →  feature vector: {feat.shape}")
print(f"Expected feature_dim = {cfg.feature_dim}")
assert feat.shape == (T_test, cfg.feature_dim), \
    f"Feature dim mismatch: {feat.shape[1]} vs {cfg.feature_dim}"
print("✓  Normalization pipeline OK")


In [ ]:
"""
🎯 POSE PROCESSING - All Exercises

⚠️  SKIP THIS CELL if processed data is already available

This cell processes raw poses into normalized features with:
- Hip-centered normalization
- Scale normalization  
- Joint angle computation
- Velocity features
- Fixed-length interpolation

Only runs processing if processed data is missing or if you want to overwrite.
"""

# Check processed data availability
processed_status = {}
for exercise in ["OHP", "Squat", "BarbellRow"]:
    processed_dir = PROCESSED_ROOT / exercise
    existing = len(list(processed_dir.glob("*.npy"))) if processed_dir.exists() else 0
    total = len(ALL_LABELS[exercise])
    processed_status[exercise] = {"existing": existing, "total": total}
    print(f"📊 {exercise} processed: {existing}/{total}")

# Configuration
skip_processing = False  # Set to True to skip processing entirely  
overwrite_processed = False  # Set to True to overwrite existing processed data

all_complete = all(status["existing"] >= status["total"] for status in processed_status.values())

if skip_processing:
    print("\n⏭️  SKIPPING pose processing (skip_processing=True)")
elif all_complete and not overwrite_processed:
    print(f"\n✅ All processed data already complete!")
    print("    Set overwrite_processed=True above to reprocess")
else:
    print(f"\n🚀 Starting pose processing (overwrite={overwrite_processed})")
    
    for exercise in ["OHP", "Squat", "BarbellRow"]:
        status = processed_status[exercise]
        if status["existing"] < status["total"] or overwrite_processed:
            print(f"\n🔄 Processing {exercise}...")
            result = process_all_poses(exercise, overwrite=overwrite_processed)
            print(f"   ✨ Processed {len(result)} {exercise} sequences")
        else:
            print(f"✅ {exercise} processing already complete ({status['existing']}/{status['total']})")
            
    print("\n🎉 All pose processing complete!")

print(f"\n📁 Processed data location: {PROCESSED_ROOT}")
print(f"💡 TIP: Processed features have shape (T={cfg.fixed_len}, F={cfg.feature_dim})")
print("       Features include: poses + joint angles + velocities")

---
## Augmentation, Datasets & Balanced Sampling

**Augmentation** (applied to all training samples):
1. Temporal jitter (±20%)
2. Gaussian noise (σ=0.01)
3. Horizontal flip (L↔R landmark swap)
4. Scale jitter (±15%)
5. Y-axis rotation (±15°)
6. Landmark dropout (1–4 landmarks)

**Two dataset classes:**
- `BalancedPoseDataset` — multi-label dataset with minority oversampling (used for eval loaders)
- `BinaryPoseDataset` — per-class balanced 1:1 dataset (used for binary classifier training)

In [ ]:
"""
Augmentation strategies for imbalanced error classes.

Approach: augment ONLY the minority (error=1) samples during training.
Never augment val/test sets.

Techniques:
  1. Temporal jitter    — random speed-up / slow-down (±20%)
  2. Gaussian noise     — small perturbation to landmark coords
  3. Mirror/flip        — horizontal flip (left↔right landmarks swapped)
  4. Scale jitter       — random global scale of the whole pose (±15%)
  5. Y-axis rotation    — small in-plane rotation around the vertical axis (±15°)
  6. Landmark dropout   — randomly zero out a subset of landmark positions
  7. Oversampling       — repeat minority keys in the dataset index
"""
import numpy as np
import torch
from torch.utils.data import Dataset, WeightedRandomSampler
from typing import Dict, List
from scipy.interpolate import interp1d


def interpolate_sequence(features: np.ndarray, target_len: int = 100) -> np.ndarray:
    """
    Resample a (T, F) feature array to (target_len, F) using linear interpolation.

    Parameters
    ----------
    features   : (T, F) float32
    target_len : int

    Returns
    -------
    resampled  : (target_len, F) float32
    """
    T = features.shape[0]
    if T == target_len:
        return features
    if T == 1:
        return np.repeat(features, target_len, axis=0)

    src_t = np.linspace(0, 1, T)
    dst_t = np.linspace(0, 1, target_len)
    interp_fn = interp1d(src_t, features, axis=0, kind="linear", fill_value="extrapolate")
    return interp_fn(dst_t).astype(np.float32)


# ── MediaPipe left↔right landmark swap map for horizontal flip ────────────────
# Format: (left_idx, right_idx) — both get swapped simultaneously
LR_SWAP_PAIRS = [
    (1, 4),   # eye inner
    (2, 5),   # eye
    (3, 6),   # eye outer
    (7, 8),   # ear
    (9, 10),  # mouth
    (11, 12), # shoulder
    (13, 14), # elbow
    (15, 16), # wrist
    (17, 18), # pinky
    (19, 20), # index
    (21, 22), # thumb
    (23, 24), # hip
    (25, 26), # knee
    (27, 28), # ankle
    (29, 30), # heel
    (31, 32), # foot index
]

def augment_landmarks(landmarks: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """
    Apply random augmentation to a (T, 33, 3) landmark array.
    All operations happen in normalised coordinate space BEFORE feature building.

    Augmentations applied (each independently gated by a probability):
      1. Temporal jitter   (p=0.8) — stretch/compress duration by ±20%
      2. Gaussian noise    (p=0.5) — small coord perturbation σ=0.01
      3. Horizontal flip   (p=0.5) — mirror + swap L/R landmark indices
      4. Scale jitter      (p=0.5) — global scale by U(0.85, 1.15)
      5. Y-axis rotation   (p=0.4) — rotate x/z by ±15° around vertical
      6. Landmark dropout  (p=0.3) — zero out 1–4 random landmark positions
    """
    aug = landmarks.copy()

    # 1. Temporal jitter: resample to random length then back
    T = aug.shape[0]
    if rng.random() < 0.8:
        factor   = rng.uniform(0.8, 1.2)
        T_new    = max(int(T * factor), 10)
        src_t    = np.linspace(0, 1, T)
        dst_t    = np.linspace(0, 1, T_new)
        aug_flat = aug.reshape(T, -1)
        fn       = interp1d(src_t, aug_flat, axis=0, kind="linear", fill_value="extrapolate")
        aug      = fn(dst_t).reshape(T_new, 33, 3).astype(np.float32)

    # 2. Gaussian noise on coordinates
    if rng.random() < 0.5:
        aug += rng.normal(0, 0.01, aug.shape).astype(np.float32)

    # 3. Horizontal flip (mirror) — swap left/right landmarks + negate x
    if rng.random() < 0.5:
        aug[:, :, 0] *= -1
        for l_idx, r_idx in LR_SWAP_PAIRS:
            aug[:, [l_idx, r_idx], :] = aug[:, [r_idx, l_idx], :]

    # 4. Scale jitter — uniform global scale around origin
    if rng.random() < 0.5:
        scale = rng.uniform(0.85, 1.15)
        aug  *= scale

    # 5. Y-axis rotation — rotate x/z coords by a small random angle
    #    (simulates slight camera angle deviation)
    if rng.random() < 0.4:
        angle = rng.uniform(-15, 15) * np.pi / 180.0
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        x_new = cos_a * aug[:, :, 0] - sin_a * aug[:, :, 2]
        z_new = sin_a * aug[:, :, 0] + cos_a * aug[:, :, 2]
        aug[:, :, 0] = x_new
        aug[:, :, 2] = z_new

    # 6. Landmark dropout — zero out a few whole landmark positions
    if rng.random() < 0.3:
        n_drop   = int(rng.integers(1, 5))     # 1–4 landmarks
        drop_idx = rng.choice(33, size=n_drop, replace=False)
        aug[:, drop_idx, :] = 0.0

    return aug


class BalancedPoseDataset(Dataset):
    """
    Pose dataset for AQA training:
      - Loads RAW landmark .npy files (T, 33, 3) from raw_pose_root
      - Builds features on-the-fly via build_feature_vector (normalise + angles + velocity)
      - Applies augmentation to ALL samples during training (not just positives),
        giving the model more diverse views; oversampling repeats positives further
      - Supports oversampling of minority class via index repetition

    Expects raw_pose_root/{exercise}/{rep_key}.npy to contain RAW landmarks.
    (run extract_all_poses / extract_all_poses_torch before using this)
    """

    def __init__(
        self,
        rep_keys:       List[str],
        labels_dict:    Dict,
        raw_pose_root:  Path,          # directory of RAW (T,33,3) .npy files
        exercise:       str,
        fixed_len:      int   = 100,
        use_angles:     bool  = True,
        use_velocity:   bool  = True,
        augment:        bool  = False,  # only True for train split
        oversample_factor: int = 3,     # repeat minority samples N times in index
        seed:           int   = 42,
    ):
        self.fixed_len    = fixed_len
        self.use_angles   = use_angles
        self.use_velocity = use_velocity
        self.augment      = augment
        self.data_dir     = raw_pose_root / exercise
        self.labels       = labels_dict
        self.rng          = np.random.default_rng(seed)

        valid_keys = [
            k for k in rep_keys
            if k in labels_dict and (self.data_dir / f"{k}.npy").exists()
        ]
        skipped = len(rep_keys) - len(valid_keys)
        if skipped:
            print(f"  [{exercise}] {skipped} keys missing pose file — skipped.")

        # ── Oversample minority class for training ─────────────────────────
        if augment and oversample_factor > 1:
            error_positive = [k for k in valid_keys if labels_dict[k].multihot.any()]
            error_negative = [k for k in valid_keys if not labels_dict[k].multihot.any()]

            oversampled_pos = error_positive * oversample_factor
            self.keys = error_negative + oversampled_pos

            print(f"  [{exercise}] Before oversampling : {len(error_negative)} neg, "
                  f"{len(error_positive)} pos")
            print(f"  [{exercise}] After  oversampling : {len(error_negative)} neg, "
                  f"{len(oversampled_pos)} pos  (×{oversample_factor})")
        else:
            self.keys = valid_keys

    def __len__(self) -> int:
        return len(self.keys)

    def __getitem__(self, idx: int):
        rep_key = self.keys[idx]
        label   = self.labels[rep_key]

        landmarks = np.load(self.data_dir / f"{rep_key}.npy")   # (T, 33, 3)

        # Augment all samples during training (not just error-positives),
        # giving the model more varied views of every rep
        if self.augment:
            landmarks = augment_landmarks(landmarks, self.rng)

        # Build feature vector: normalise + angles + velocity
        features = build_feature_vector(
            landmarks,
            use_angles   = self.use_angles,
            use_velocity = self.use_velocity,
        )                                                        # (T, F)

        features = interpolate_sequence(features, self.fixed_len)  # (fixed_len, F)

        return {
            "X":        torch.from_numpy(features),
            "y_score":  torch.tensor(label.score,    dtype=torch.float32),
            "y_errors": torch.from_numpy(label.multihot),
            "rep_key":  rep_key,
        }


def make_weighted_sampler(keys: List[str], labels_dict: Dict) -> WeightedRandomSampler:
    """
    Per-sample weights inversely proportional to class frequency.
    Ensures each batch sees roughly equal positive/negative samples.
    """
    n_errors = len(next(iter(labels_dict.values())).multihot)

    # Count positives per class
    n_pos = np.zeros(n_errors)
    for k in keys:
        n_pos += labels_dict[k].multihot

    n_total = len(keys)
    n_neg   = n_total - n_pos

    # Weight per sample = sum of per-class weights for its error flags
    sample_weights = []
    for k in keys:
        mh = labels_dict[k].multihot
        w  = 0.0
        for i in range(n_errors):
            if mh[i] == 1:
                w += n_total / (n_pos[i] + 1e-6)
            else:
                w += n_total / (n_neg[i] + 1e-6)
        sample_weights.append(w / n_errors)

    return WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float32),
        num_samples = len(sample_weights),
        replacement = True,
    )


# ── Class balance summary for all exercises ───────────────────────────────────
print("Class imbalance summary:")
print(f"{'Exercise':<15} {'Error class':<28} {'Neg':>6} {'Pos':>6} {'Ratio':>7}")
print("-" * 65)
for ex in EXERCISE_CONFIG:
    ld          = ALL_LABELS[ex]
    error_names = list(EXERCISE_CONFIG[ex]["label_files"].keys())
    for i, ename in enumerate(error_names):
        n_pos = sum(1 for r in ld.values() if r.multihot[i] == 1)
        n_neg = len(ld) - n_pos
        ratio = n_neg / max(n_pos, 1)
        flag  = "  ← high imbalance" if ratio > 4 else ""
        print(f"{ex:<15} {ename:<28} {n_neg:>6} {n_pos:>6} {ratio:>6.1f}x{flag}")

In [ ]:
"""
Per-class balanced binary dataset for the per-class binary classifier approach.

Key idea: instead of one multi-label model per exercise with complex imbalance
corrections (oversampling + WeightedRandomSampler + focal loss + pos_weight),
train one binary classifier per error class with a perfectly balanced 1:1 dataset.

This eliminates imbalance at the data level:
  - No pos_weight needed (dataset is 50/50)
  - No focal loss needed (no class is harder to sample)
  - No WeightedRandomSampler needed (dataset is already balanced)
  - Augmentation makes repeated samples appear different each time
"""
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path
from typing import Dict, List


class BinaryPoseDataset(Dataset):
    """
    Balanced binary pose dataset for a single error class.

    For training:
      - Splits keys into positive (target_class=1) and negative (target_class=0)
      - Oversamples the minority class to match the majority class
      - Applies augmentation to ALL samples (each repeated sample gets different augmentation)

    For val/test:
      - Uses all keys without balancing
      - No augmentation
    """

    def __init__(
        self,
        rep_keys:         List[str],
        labels_dict:      Dict,
        raw_pose_root:    Path,
        exercise:         str,
        target_class_idx: int,
        fixed_len:        int  = 100,
        use_angles:       bool = True,
        use_velocity:     bool = True,
        augment:          bool = False,
        seed:             int  = 42,
    ):
        self.fixed_len        = fixed_len
        self.use_angles       = use_angles
        self.use_velocity     = use_velocity
        self.augment          = augment
        self.data_dir         = raw_pose_root / exercise
        self.labels           = labels_dict
        self.target_class_idx = target_class_idx
        self.rng              = np.random.default_rng(seed)

        valid_keys = [
            k for k in rep_keys
            if k in labels_dict and (self.data_dir / f"{k}.npy").exists()
        ]
        skipped = len(rep_keys) - len(valid_keys)
        if skipped:
            print(f"  [{exercise}/class{target_class_idx}] {skipped} keys missing — skipped.")

        if augment:
            pos_keys = [k for k in valid_keys if labels_dict[k].multihot[target_class_idx] == 1]
            neg_keys = [k for k in valid_keys if labels_dict[k].multihot[target_class_idx] == 0]

            n_pos, n_neg = len(pos_keys), len(neg_keys)

            # Oversample minority to match majority
            if n_pos < n_neg and n_pos > 0:
                repeat = n_neg // n_pos + 1
                pos_keys = (pos_keys * repeat)[:n_neg]
            elif n_neg < n_pos and n_neg > 0:
                repeat = n_pos // n_neg + 1
                neg_keys = (neg_keys * repeat)[:n_pos]

            self.keys = neg_keys + pos_keys
            print(f"  [{exercise}/class{target_class_idx}] Balanced: "
                  f"{len(neg_keys)} neg + {len(pos_keys)} pos = {len(self.keys)} total "
                  f"(from {n_neg} unique neg, {n_pos} unique pos)")
        else:
            self.keys = valid_keys

    def __len__(self) -> int:
        return len(self.keys)

    def __getitem__(self, idx: int):
        rep_key = self.keys[idx]
        label   = self.labels[rep_key]

        landmarks = np.load(self.data_dir / f"{rep_key}.npy")   # (T, 33, 3)

        if self.augment:
            landmarks = augment_landmarks(landmarks, self.rng)

        features = build_feature_vector(
            landmarks,
            use_angles=self.use_angles,
            use_velocity=self.use_velocity,
        )
        features = interpolate_sequence(features, self.fixed_len)

        return {
            "X":        torch.from_numpy(features),
            "y_binary": torch.tensor(label.multihot[self.target_class_idx], dtype=torch.float32),
            "y_errors": torch.from_numpy(label.multihot),
            "y_score":  torch.tensor(label.score, dtype=torch.float32),
            "rep_key":  rep_key,
        }


print("✓ BinaryPoseDataset defined (per-class balanced binary dataset)")

---
## DataLoader Factory

In [ ]:
"""
PyTorch DataLoader factory.

interpolate_sequence, BalancedPoseDataset, and make_weighted_sampler
are defined in the augmentation cell above.
"""
import os
import tempfile
import numpy as np
import torch
from torch.utils.data import DataLoader
from pathlib import Path
from typing import Dict, Optional


def recommended_num_workers() -> int:
    """Use more workers when CUDA is available; keep 0 on CPU-only for stability."""
    if not torch.cuda.is_available():
        return 0
    return min(8, os.cpu_count() or 2)


# ── DataLoader factory ─────────────────────────────────────────────────────────
def make_dataloader(
    split:          str,
    exercise:       str,
    labels_dict:    Dict,
    all_splits:     Dict,
    raw_pose_root:  Path = POSE_ROOT,
    batch_size:     int  = 32,
    fixed_len:      int  = 100,
    use_angles:     bool = True,
    use_velocity:   bool = True,
    use_weighted_sampler: bool = True,  # train split only
    oversample_factor:    int  = 3,     # train split only
    num_workers:    Optional[int]  = None,
    persistent_workers: Optional[bool] = None,
    prefetch_factor: Optional[int] = None,
) -> DataLoader:
    is_train = (split == "train")

    ds = BalancedPoseDataset(
        rep_keys          = all_splits[split],
        labels_dict       = labels_dict,
        raw_pose_root     = raw_pose_root,
        exercise          = exercise,
        fixed_len         = fixed_len,
        use_angles        = use_angles,
        use_velocity      = use_velocity,
        augment           = is_train,
        oversample_factor = oversample_factor if is_train else 1,
    )

    sampler = None
    if is_train and use_weighted_sampler and len(ds) > 0:
        sampler = make_weighted_sampler(ds.keys, labels_dict)

    if num_workers is None:
        num_workers = recommended_num_workers()
    if persistent_workers is None:
        persistent_workers = num_workers > 0
    if prefetch_factor is None and num_workers > 0:
        prefetch_factor = 2

    loader_kwargs = dict(
        dataset     = ds,
        batch_size  = batch_size,
        shuffle     = is_train and (sampler is None),
        sampler     = sampler,
        num_workers = num_workers,
        pin_memory  = torch.cuda.is_available(),
        drop_last   = is_train,
        persistent_workers = persistent_workers,
    )
    if num_workers > 0 and prefetch_factor is not None:
        loader_kwargs["prefetch_factor"] = prefetch_factor

    loader = DataLoader(**loader_kwargs)
    sampler_name = "WeightedRandomSampler" if sampler is not None else "none"
    print(
        f"[{exercise}] {split:5s}: {len(ds)} samples → {len(loader)} batches "
        f"(sampler={sampler_name}, workers={num_workers})"
    )
    return loader


# ── Smoke test with synthetic landmark data ────────────────────────────────────
def _make_fake_dataset(exercise: str = "OHP", n: int = 20, T: int = 73):
    """Write synthetic raw landmark .npy files and fake labels for testing."""
    from dataclasses import dataclass

    tmpdir = Path(tempfile.mkdtemp()) / exercise
    tmpdir.mkdir(parents=True)

    keys, labels = [], {}
    for i in range(n):
        key = f"99999_{i}"
        landmarks = np.random.randn(T, 33, 3).astype(np.float32)  # raw (T, 33, 3)
        np.save(tmpdir / f"{key}.npy", landmarks)

        @dataclass
        class _L:
            rep_key: str; subject_id: str; exercise: str
            errors: list; intervals: dict; score: float; multihot: np.ndarray

        labels[key] = _L(key, "99999", exercise, [], {}, float(np.random.rand()),
                         np.random.randint(0, 2, 2).astype(np.float32))
        keys.append(key)
    return tmpdir.parent, keys, labels


fake_root, fake_keys, fake_labels = _make_fake_dataset()
splits_fake = {"train": fake_keys[:14], "val": fake_keys[14:17], "test": fake_keys[17:]}

train_loader = make_dataloader(
    "train", "OHP", fake_labels, splits_fake,
    raw_pose_root=fake_root, batch_size=4,
    fixed_len=cfg.fixed_len,
    use_angles=cfg.use_angles, use_velocity=cfg.use_velocity,
    num_workers=0,
)
batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  X        : {batch['X'].shape}")
print(f"  y_score  : {batch['y_score'].shape}")
print(f"  y_errors : {batch['y_errors'].shape}")
assert batch["X"].shape == (4, cfg.fixed_len, cfg.feature_dim), \
    f"Unexpected shape: {batch['X'].shape}"
print("✓  BalancedPoseDataset + DataLoader OK")

In [ ]:
"""
DataLoader factory for per-class binary classifiers.
No WeightedRandomSampler — the dataset itself is balanced 1:1.
"""
from torch.utils.data import DataLoader


def make_binary_dataloader(
    split:            str,
    exercise:         str,
    labels_dict:      Dict,
    all_splits:       Dict,
    target_class_idx: int,
    raw_pose_root:    Path = POSE_ROOT,
    batch_size:       int  = 32,
    fixed_len:        int  = 100,
    use_angles:       bool = True,
    use_velocity:     bool = True,
    num_workers:      int | None = None,
) -> DataLoader:
    is_train = (split == "train")

    ds = BinaryPoseDataset(
        rep_keys         = all_splits[split],
        labels_dict      = labels_dict,
        raw_pose_root    = raw_pose_root,
        exercise         = exercise,
        target_class_idx = target_class_idx,
        fixed_len        = fixed_len,
        use_angles       = use_angles,
        use_velocity     = use_velocity,
        augment          = is_train,
    )

    if num_workers is None:
        num_workers = recommended_num_workers()

    loader_kwargs = dict(
        dataset            = ds,
        batch_size         = batch_size,
        shuffle            = is_train,
        num_workers        = num_workers,
        pin_memory         = torch.cuda.is_available(),
        drop_last          = is_train and len(ds) > batch_size,
        persistent_workers = num_workers > 0,
    )
    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = 2

    loader = DataLoader(**loader_kwargs)
    print(f"  [{exercise}] {split:5s} class{target_class_idx}: "
          f"{len(ds)} samples → {len(loader)} batches")
    return loader


print("✓ make_binary_dataloader defined (no WeightedRandomSampler — data is balanced)")

---
## Stage 7 — Subject-Aware Dataset Splitting

The pre-provided splits already partition by **subject ID**, so we respect and verify them.
We also provide a `build_subject_split()` utility for any custom exercise or new data.

> **Data leakage rule:** a subject must appear in exactly one of train/val/test.

In [ ]:
"""
Subject-aware split utilities.
"""
import random
import numpy as np
from collections import defaultdict
from typing import Dict, List, Tuple


def verify_subject_splits(
    exercise: str,
    labels_dict: Dict,
    splits: Dict[str, List[str]],
    verbose: bool = True,
) -> bool:
    """
    Verify that subjects in train/val/test are non-overlapping.
    Returns True if no leakage detected.
    """
    split_subjects = {}
    for split_name, keys in splits.items():
        subjs = {labels_dict[k].subject_id for k in keys if k in labels_dict}
        split_subjects[split_name] = subjs

    ok = True
    checked = []
    for i, (s1, subj1) in enumerate(split_subjects.items()):
        for s2, subj2 in list(split_subjects.items())[i+1:]:
            overlap = subj1 & subj2
            if overlap:
                print(f"  ⚠  LEAKAGE: {s1} ∩ {s2} = {len(overlap)} subjects")
                ok = False
            elif verbose:
                print(f"  ✓  {s1} ∩ {s2} = 0 subjects (clean)")

    if verbose:
        for split_name, subjs in split_subjects.items():
            print(f"  {exercise} {split_name:5s}: {len(splits[split_name]):4d} reps, "
                  f"{len(subjs):3d} subjects")
    return ok


def build_subject_split(
    labels_dict:   Dict,
    train_frac:    float = 0.70,
    val_frac:      float = 0.15,
    seed:          int   = 42,
) -> Dict[str, List[str]]:
    """
    Build a subject-level random split from scratch.

    Groups all reps by subject, then assigns each subject (not each rep)
    to a partition.  Returns {'train': [...], 'val': [...], 'test': [...]}.
    """
    rng = random.Random(seed)

    # group rep_keys by subject
    subject_map: Dict[str, List[str]] = defaultdict(list)
    for key, label in labels_dict.items():
        subject_map[label.subject_id].append(key)

    subjects = sorted(subject_map.keys())
    rng.shuffle(subjects)

    n    = len(subjects)
    n_tr = int(n * train_frac)
    n_va = int(n * val_frac)

    train_subjs = subjects[:n_tr]
    val_subjs   = subjects[n_tr: n_tr + n_va]
    test_subjs  = subjects[n_tr + n_va:]

    result = {
        "train": [k for s in train_subjs for k in subject_map[s]],
        "val":   [k for s in val_subjs   for k in subject_map[s]],
        "test":  [k for s in test_subjs  for k in subject_map[s]],
    }

    print(f"Built subject split:  train={len(result['train'])}  "
          f"val={len(result['val'])}  test={len(result['test'])}")
    print(f"  subjects:  train={len(train_subjs)}  val={len(val_subjs)}  test={len(test_subjs)}")
    return result


# ── Verify official splits ─────────────────────────────────────────────────────
print("=" * 55)
for ex in EXERCISE_CONFIG:
    print(f"\n── {ex} ──")
    verify_subject_splits(ex, ALL_LABELS[ex], ALL_SPLITS[ex])


---
## Temporal Convolutional Network (TCN)

**Classification-only architecture** (no regression head — score derived post-hoc):

```
Input (B, T, F)
     ↓  Linear projection → (B, T, hidden)
     ↓  transpose → (B, hidden, T)
     ↓  TCN blocks × N  [dilated causal Conv1D + residual + BatchNorm + dropout]
     ↓  Attention Pool → (B, hidden)  [learned soft attention over time]
     └── Classification head → logits (B, n_error_classes)
```

- **Dilated causal convolutions** capture multi-scale temporal patterns (receptive field covers all 100 frames)
- **AttentionPool** focuses on the most discriminative frames rather than averaging uniformly
- One model per error class with `n_error_classes=1` (binary)

In [ ]:
"""
Classification-only Temporal Convolutional Network (TCN) for AQA.

The regression head is removed — quality score can be derived post-hoc
from the predicted error probabilities (e.g. 1 - mean_error_prob).
Architecture: hidden_dim=256, n_layers=8, dropout=0.3,
              AttentionPool (replaces global average pool for better
              intra-sequence focus on key frames).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


# ── Causal TCN block ───────────────────────────────────────────────────────────
class CausalConv1dBlock(nn.Module):
    """
    Single TCN residual block with dilated causal convolution.

    padding = (kernel_size - 1) * dilation  →  causal (no future leakage)
    """

    def __init__(
        self,
        in_channels:  int,
        out_channels: int,
        kernel_size:  int = 3,
        dilation:     int = 1,
        dropout:      float = 0.3,
    ):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=pad, dilation=dilation)
        self.norm1  = nn.BatchNorm1d(out_channels)
        self.norm2  = nn.BatchNorm1d(out_channels)
        self.drop   = nn.Dropout(dropout)
        self.pad    = pad

        # 1×1 projection for residual if channel dims differ
        self.shortcut = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def _causal_trim(self, x: torch.Tensor) -> torch.Tensor:
        """Remove future padding added by conv."""
        return x[:, :, :-self.pad] if self.pad > 0 else x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        residual = self.shortcut(x)
        out = F.relu(self.norm1(self._causal_trim(self.conv1(x))))
        out = self.drop(out)
        out = F.relu(self.norm2(self._causal_trim(self.conv2(out))))
        out = self.drop(out)
        return F.relu(out + residual)


# ── Attention pooling ──────────────────────────────────────────────────────────
class AttentionPool(nn.Module):
    """
    Soft attention pooling over the time axis.

    Learns a query vector to weight each time step before summing,
    allowing the model to focus on the most discriminative frames
    (e.g. the moment an error occurs) rather than averaging uniformly.
    """

    def __init__(self, hidden_dim: int, attn_dim: int = 64):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim),
            nn.Tanh(),
            nn.Linear(attn_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, hidden, T)
        x_t     = x.transpose(1, 2)                      # (B, T, hidden)
        scores  = self.attn(x_t).squeeze(-1)              # (B, T)
        weights = torch.softmax(scores, dim=1)             # (B, T)
        return (x_t * weights.unsqueeze(-1)).sum(dim=1)   # (B, hidden)


# ── Full TCN ───────────────────────────────────────────────────────────────────
class ExerciseTCN(nn.Module):
    """
    Classification-only TCN for exercise error detection.

    The regression head is intentionally omitted — a quality score can be
    derived post-hoc as:  score = 1 - sigmoid(logits).mean(axis=-1)

    Parameters
    ----------
    input_dim       : F (feature dimension per frame)
    n_error_classes : number of binary error labels
    hidden_dim      : channel width for TCN blocks (default 256)
    n_layers        : number of TCN blocks; dilation doubles each layer (default 8)
    kernel_size     : convolution kernel size (default 3)
    dropout         : dropout rate (default 0.3)
    """

    def __init__(
        self,
        input_dim:       int,
        n_error_classes: int,
        hidden_dim:      int   = 256,
        n_layers:        int   = 8,
        kernel_size:     int   = 3,
        dropout:         float = 0.3,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        layers = []
        for i in range(n_layers):
            dilation = 2 ** i               # 1, 2, 4, 8, 16, 32, 64, 128 …
            layers.append(CausalConv1dBlock(
                in_channels  = hidden_dim,
                out_channels = hidden_dim,
                kernel_size  = kernel_size,
                dilation     = dilation,
                dropout      = dropout,
            ))
        self.tcn = nn.Sequential(*layers)

        # Attention pooling over time (better than global average pool)
        self.attn_pool = AttentionPool(hidden_dim)

        # ── Classification head ───────────────────────────────────────────────
        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_error_classes),
            # no activation — BCEWithLogitsLoss handles sigmoid internally
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (B, T, F)

        Returns
        -------
        logits : (B, n_error_classes)  — raw logits; apply sigmoid for probabilities
        """
        h = self.input_proj(x)              # (B, T, hidden)
        h = h.transpose(1, 2)              # (B, hidden, T) — Conv1d expects (B, C, L)
        h = self.tcn(h)                    # (B, hidden, T)
        h = self.attn_pool(h)              # Attention pool → (B, hidden)
        return self.cls_head(h)            # (B, n_error_classes)


# ── Sanity check ───────────────────────────────────────────────────────────────
batch_size  = 8
seq_len     = cfg.fixed_len
feature_dim = cfg.feature_dim
n_errors    = len(EXERCISE_CONFIG["OHP"]["label_files"])    # 2 for OHP

model   = ExerciseTCN(input_dim=feature_dim, n_error_classes=n_errors,
                      hidden_dim=cfg.hidden_dim, n_layers=cfg.n_layers,
                      dropout=cfg.dropout)
dummy   = torch.randn(batch_size, seq_len, feature_dim)
logits  = model(dummy)

print(f"Model input    : {dummy.shape}")
print(f"Logit output   : {logits.shape}  (values: {logits.min():.3f}–{logits.max():.3f})")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters : {total_params:,}")

print(f"✓  TCN model OK  (classification-only, hidden={cfg.hidden_dim}, "
      f"layers={cfg.n_layers}, dropout={cfg.dropout}, AttentionPool)")

---
## Training — Per-Class Binary Classifiers

**Approach:** One binary classifier per error class, each trained on a balanced 1:1 dataset.

**Loss:** `BCEWithLogitsLoss` (standard — no pos_weight, no focal loss, no WeightedRandomSampler needed)

**Training details:**
- AdamW optimiser, `lr=3e-4`, `weight_decay=5e-4`
- ReduceLROnPlateau scheduler (patience=5, factor=0.5)
- Early stopping patience=30, AMP (mixed precision) on CUDA
- DataParallel across available GPUs
- Per-class threshold sweep on val set (precision floor = 0.20)
- Checkpoints saved to `checkpoints/{exercise}_{class_name}_best.pt`

In [ ]:
"""
Per-class binary classifier training.

Approach: one binary classifier per error class, each trained on a balanced
1:1 dataset. Uses standard BCE loss — no focal loss, no pos_weight, no
WeightedRandomSampler needed since the data is perfectly balanced.

  • Eliminates all complex imbalance correction interactions
  • Each model specialises on detecting one specific error type
  • Augmentation ensures repeated minority samples appear different each time
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from pathlib import Path
from typing import Dict, Tuple, List
from sklearn.metrics import f1_score as sk_f1_score, precision_score as sk_precision_score
import copy, time, math

CHECKPOINT_DIR = Path(os.environ.get("EXERCISE_ADVISOR_CHECKPOINTS_LEGACY", "../data/checkpoints_legacy"))
CHECKPOINT_DIR.mkdir(exist_ok=True)


# ── Early stopping ─────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = cfg.patience, min_delta: float = 1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = float("inf")
        self.counter   = 0
        self.best_state: Dict = {}
        self.triggered = False

    def step(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best - self.min_delta:
            self.best       = val_loss
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered

    def restore_best(self, model: nn.Module):
        if self.best_state:
            model.load_state_dict(self.best_state)


# ── Main per-class binary training function ────────────────────────────────────
def train_binary(
    exercise:       str,
    class_idx:      int,
    class_name:     str,
    cfg:            "Config"         = cfg,
    device_str:     str              = "auto",
    use_amp:        bool             = True,
    use_multi_gpu:  bool             = True,
    gpu_ids:        List[int] | None = None,
    loader_workers: int | None       = None,
) -> Tuple[nn.Module, Dict]:
    """
    Train one binary classifier for a single error class.
    Dataset is balanced 1:1 → standard BCE loss, no class weighting needed.
    """
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    ) if device_str == "auto" else torch.device(device_str)

    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    labels_dict = ALL_LABELS[exercise]
    splits      = ALL_SPLITS[exercise]

    # ── Binary-balanced DataLoaders ────────────────────────────────────────
    loaders = {
        split: make_binary_dataloader(
            split, exercise, labels_dict, splits, class_idx,
            batch_size=cfg.batch_size, fixed_len=cfg.fixed_len,
            use_angles=cfg.use_angles, use_velocity=cfg.use_velocity,
            num_workers=loader_workers,
        )
        for split in ["train", "val"]
    }

    # ── Model (1 output neuron) ────────────────────────────────────────────
    model = ExerciseTCN(
        input_dim=cfg.feature_dim, n_error_classes=1,
        hidden_dim=cfg.hidden_dim, n_layers=cfg.n_layers,
        kernel_size=3, dropout=cfg.dropout,
    ).to(device)

    cuda_count       = torch.cuda.device_count() if device.type == "cuda" else 0
    selected_gpu_ids = list(range(cuda_count)) if gpu_ids is None else gpu_ids
    if device.type == "cuda" and use_multi_gpu and len(selected_gpu_ids) > 1:
        model = nn.DataParallel(model, device_ids=selected_gpu_ids)
        print(f"    DataParallel on GPUs {selected_gpu_ids}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Model params: {n_params:,}")

    # ── Standard BCE loss — data is balanced, no weighting needed ──────────
    criterion = nn.BCEWithLogitsLoss()

    optimiser = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    try:
        scheduler = ReduceLROnPlateau(optimiser, patience=5, factor=0.5, verbose=True)
    except TypeError:
        scheduler = ReduceLROnPlateau(optimiser, patience=5, factor=0.5)

    stopper     = EarlyStopping(patience=cfg.patience)
    amp_enabled = use_amp and device.type == "cuda"
    scaler      = torch.amp.GradScaler(device="cuda", enabled=amp_enabled)

    history: Dict[str, List] = {"train_loss": [], "val_loss": [], "lr": []}

    for epoch in range(1, cfg.max_epochs + 1):
        t0 = time.time()

        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        tr_total, tr_n = 0.0, 0
        for batch in loaders["train"]:
            X = batch["X"].to(device, non_blocking=True)
            y = batch["y_binary"].to(device, non_blocking=True).unsqueeze(1)  # (B, 1)

            optimiser.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(X)   # (B, 1)
                loss   = criterion(logits, y)

            if not torch.isfinite(loss):
                optimiser.zero_grad(set_to_none=True)
                continue

            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimiser)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimiser)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimiser.step()

            tr_total += loss.item()
            tr_n     += 1

        tr_loss = tr_total / max(tr_n, 1)

        # ── Val ────────────────────────────────────────────────────────────
        model.eval()
        vl_total, vl_n = 0.0, 0
        with torch.no_grad():
            for batch in loaders["val"]:
                X = batch["X"].to(device, non_blocking=True)
                y = batch["y_binary"].to(device, non_blocking=True).unsqueeze(1)
                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    logits = model(X)
                    loss   = criterion(logits, y)
                if torch.isfinite(loss):
                    vl_total += loss.item()
                    vl_n     += 1
        vl_loss = vl_total / max(vl_n, 1)

        if math.isnan(tr_loss) or math.isnan(vl_loss):
            print(f"    NaN at epoch {epoch} — aborting")
            break

        scheduler.step(vl_loss)
        lr_now = optimiser.param_groups[0]["lr"]
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["lr"].append(lr_now)

        elapsed = time.time() - t0
        if epoch == 1 or epoch % 10 == 0:
            print(f"    Epoch {epoch:3d}: train={tr_loss:.4f}  val={vl_loss:.4f}  "
                  f"lr={lr_now:.2e}  ({elapsed:.1f}s)")

        if stopper.step(vl_loss, model):
            print(f"    Early stopping at epoch {epoch}  (best val={stopper.best:.4f})")
            break

    stopper.restore_best(model)

    # ── Threshold sweep on val set ─────────────────────────────────────────
    best_threshold = 0.5
    try:
        model.eval()
        _all_logits, _all_labels = [], []
        with torch.no_grad():
            for _b in loaders["val"]:
                _X = _b["X"].to(device)
                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    _lgts = model(_X)
                _all_logits.append(_lgts.cpu().float().numpy().ravel())
                _all_labels.append(_b["y_binary"].numpy().ravel())

        _all_logits = np.concatenate(_all_logits)
        _all_labels = np.concatenate(_all_labels).astype(int)
        _all_probs  = 1.0 / (1.0 + np.exp(-_all_logits))

        _best_f1 = -1.0
        for _t in np.arange(0.05, 0.96, 0.05):
            _preds = (_all_probs >= _t).astype(int)
            _prec  = sk_precision_score(_all_labels, _preds, zero_division=0.0)
            if _prec < 0.20:       # precision floor
                continue
            _f1 = sk_f1_score(_all_labels, _preds, zero_division=0)
            if _f1 > _best_f1:
                _best_f1, best_threshold = _f1, float(_t)

        if _best_f1 < 0:
            best_threshold = 0.5
            print(f"    No threshold exceeded precision floor — using 0.5")
        print(f"    Best val threshold: {best_threshold:.2f} (F1={_best_f1:.3f})")
    except Exception as _e:
        print(f"    Threshold sweep failed ({_e}) — using 0.5")

    history["best_threshold"] = best_threshold

    # ── Checkpoint ─────────────────────────────────────────────────────────
    model_to_save = model.module if isinstance(model, nn.DataParallel) else model
    ckpt_path     = CHECKPOINT_DIR / f"{exercise}_{class_name}_best.pt"
    torch.save({
        "model_state":    model_to_save.state_dict(),
        "config":         cfg,
        "class_name":     class_name,
        "class_idx":      class_idx,
        "exercise":       exercise,
        "history":        history,
        "best_threshold": best_threshold,
    }, ckpt_path)
    print(f"    Checkpoint → {ckpt_path.name}")

    return model, history


print("Per-class binary training function defined (train_binary).")
print(f"  Loss       : BCEWithLogitsLoss (standard — data is balanced)")
print(f"  Model      : ExerciseTCN(n_error_classes=1) per class")
print(f"  Architecture: hidden={cfg.hidden_dim}, layers={cfg.n_layers}, dropout={cfg.dropout}")
print(f"  LR={cfg.lr}, weight_decay={cfg.weight_decay}, patience={cfg.patience}")

In [ ]:
"""
Train per-class binary classifiers for ALL exercises and ALL error labels.
One model per error class = completely balanced training.
Models moved to CPU after training to free GPU memory.
Skips any model that already has a checkpoint (set FORCE_RETRAIN=True to override).
"""
import torch, gc
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

FORCE_RETRAIN = False   # Set True to retrain even if checkpoint exists

# ── Train ─────────────────────────────────────────────────────────────────────
trained_models = {}   # {exercise: {class_name: model}}
histories      = {}   # {exercise: {class_name: history}}

for exercise in ["OHP", "Squat", "BarbellRow"]:
    error_names = list(EXERCISE_CONFIG[exercise]["label_files"].keys())
    trained_models[exercise] = {}
    histories[exercise]      = {}

    print(f"\n{'='*60}")
    print(f"  {exercise}: {len(error_names)} error classes → {error_names}")
    print(f"{'='*60}")

    for ci, cname in enumerate(error_names):
        ckpt_path = CHECKPOINT_DIR / f"{exercise}_{cname}_best.pt"

        # ── Load from checkpoint if available ──────────────────────────────
        if ckpt_path.exists() and not FORCE_RETRAIN:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            model = ExerciseTCN(
                input_dim=cfg.feature_dim, n_error_classes=1,
                hidden_dim=cfg.hidden_dim, n_layers=cfg.n_layers,
                kernel_size=3, dropout=cfg.dropout,
            )
            model.load_state_dict(ckpt["model_state"])
            trained_models[exercise][cname] = model
            histories[exercise][cname]      = ckpt["history"]
            thr = ckpt["history"].get("best_threshold", 0.5)
            print(f"  ✓ {exercise}/{cname}: loaded from checkpoint (threshold={thr:.2f})")
            continue

        # ── Train from scratch ─────────────────────────────────────────────
        print(f"\n  ── {exercise}/{cname} (class {ci}) {'─'*30}")
        model, history = train_binary(
            exercise       = exercise,
            class_idx      = ci,
            class_name     = cname,
            use_multi_gpu  = True,
            use_amp        = True,
            loader_workers = 4,
        )
        # Move model to CPU to free GPU memory for next model
        if isinstance(model, nn.DataParallel):
            model = model.module
        model = model.cpu()
        trained_models[exercise][cname] = model
        histories[exercise][cname]      = history

        best_vl = min(history['val_loss']) if history['val_loss'] else float('nan')
        print(f"    Best val loss: {best_vl:.4f}")

        # Free GPU memory
        torch.cuda.empty_cache()
        gc.collect()

n_total = sum(len(v) for v in trained_models.values())
print(f"\n✓ {n_total} per-class binary models ready.")

# ── Loss curves ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

for exercise in trained_models:
    n_classes = len(trained_models[exercise])
    fig, axes = plt.subplots(1, n_classes, figsize=(5*n_classes, 3), squeeze=False)
    fig.suptitle(f"{exercise} — Per-class Binary Loss Curves", fontweight="bold")
    for i, (cname, hist) in enumerate(histories[exercise].items()):
        ax = axes[0, i]
        ax.plot(hist["train_loss"], label="train")
        ax.plot(hist["val_loss"],   label="val")
        ax.set_title(cname)
        ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
        ax.legend()
    plt.tight_layout(); plt.show()

---
## Stage 10 — Evaluation

**Regression metrics:**
- MAE (Mean Absolute Error)  
- Spearman rank correlation $\rho$ — measures ranking quality, robust to score scale

**Classification metrics (per error class and macro-averaged):**
- Precision, Recall, F1 — threshold 0.5 on sigmoid output
- ROC-AUC per class  

**Failure analysis:** scatter high-error reps, visualise predicted vs true score.

In [ ]:
"""
Evaluation for per-class binary classifiers.
Combines predictions from individual per-class models into a multi-label
prediction matrix, then computes the same metrics as before.
"""
import torch
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, mean_absolute_error,
)
import matplotlib.pyplot as plt
from typing import Dict, List


@torch.no_grad()
def predict_all_binary(
    models_dict:     Dict[str, nn.Module],
    loader:          DataLoader,
    device:          torch.device,
    thresholds_dict: Dict[str, float],
    error_names:     List[str],
) -> Dict[str, np.ndarray]:
    """
    Combine per-class binary model predictions into multi-label format.

    Parameters
    ----------
    models_dict     : {class_name: trained_model}
    thresholds_dict : {class_name: float threshold}
    error_names     : ordered list of error class names
    """
    # Collect all data in one pass through the loader
    all_X, all_scores, all_errors, all_keys = [], [], [], []
    for batch in loader:
        all_X.append(batch["X"])
        all_scores.append(batch["y_score"])
        all_errors.append(batch["y_errors"])
        all_keys.extend(batch["rep_key"])

    X_cat       = torch.cat(all_X)                              # (N, T, F)
    true_scores = torch.cat(all_scores).numpy()                 # (N,)
    true_labels = torch.cat(all_errors).numpy().astype(int)     # (N, n_classes)

    n_samples = X_cat.shape[0]
    n_classes = len(error_names)
    pred_probs  = np.zeros((n_samples, n_classes), dtype=np.float32)
    pred_labels = np.zeros((n_samples, n_classes), dtype=int)

    for ci, cname in enumerate(error_names):
        model = models_dict[cname]
        model.eval()
        model_device = next(model.parameters()).device

        logits_list = []
        bs = 64
        for i in range(0, n_samples, bs):
            batch_X = X_cat[i:i+bs].to(model_device)
            with torch.amp.autocast(device_type=model_device.type,
                                    enabled=model_device.type == "cuda"):
                lgts = model(batch_X)
            logits_list.append(lgts.cpu().float().numpy().ravel())

        logits_c = np.concatenate(logits_list)
        probs_c  = 1.0 / (1.0 + np.exp(-logits_c))

        thresh = thresholds_dict.get(cname, 0.5)
        pred_probs[:, ci]  = probs_c
        pred_labels[:, ci] = (probs_c >= thresh).astype(int)

    derived_scores = 1.0 - pred_probs.mean(axis=1)

    return {
        "derived_scores": derived_scores,
        "true_scores":    true_scores,
        "pred_probs":     pred_probs,
        "pred_labels":    pred_labels,
        "true_labels":    true_labels,
        "rep_keys":       np.array(all_keys),
    }


def evaluate_binary(
    models_dict:     Dict[str, nn.Module],
    loader:          DataLoader,
    exercise:        str,
    split:           str  = "test",
    thresholds_dict: Dict[str, float] = None,
) -> Dict:
    """
    Full evaluation for per-class binary classifiers.
    Prints metrics table and plots.
    """
    error_names = list(EXERCISE_CONFIG[exercise]["label_files"].keys())
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if thresholds_dict is None:
        thresholds_dict = {cn: 0.5 for cn in error_names}

    preds = predict_all_binary(models_dict, loader, device, thresholds_dict, error_names)

    n_samples = len(preds["derived_scores"])
    print(f"\n{'='*60}")
    print(f"  {exercise}  [{split}]  — {n_samples} samples  (per-class binary)")
    print(f"{'='*60}")

    # ── Derived score ─────────────────────────────────────────────────────
    rho, p_val = spearmanr(preds["true_scores"], preds["derived_scores"])
    mae        = mean_absolute_error(preds["true_scores"], preds["derived_scores"])
    print(f"\n  Derived score (1 - mean_error_prob):")
    print(f"    Spearman ρ : {rho:.4f}  (p={p_val:.3e})")
    print(f"    MAE        : {mae:.4f}")

    # ── Per-class classification ──────────────────────────────────────────
    print(f"\n  Per-class binary classification:")
    for i, ename in enumerate(error_names):
        prec = precision_score(preds["true_labels"][:, i], preds["pred_labels"][:, i], zero_division=0)
        rec  = recall_score   (preds["true_labels"][:, i], preds["pred_labels"][:, i], zero_division=0)
        f1   = f1_score       (preds["true_labels"][:, i], preds["pred_labels"][:, i], zero_division=0)
        try:
            auc = roc_auc_score(preds["true_labels"][:, i], preds["pred_probs"][:, i])
        except ValueError:
            auc = float("nan")
        t = thresholds_dict.get(ename, 0.5)
        print(f"    {ename:<28}: P={prec:.2f}  R={rec:.2f}  F1={f1:.2f}  "
              f"AUC={auc:.3f}  (t={t:.2f})")

    macro_f1 = f1_score(preds["true_labels"], preds["pred_labels"],
                        average="macro", zero_division=0)
    print(f"\n    Macro F1: {macro_f1:.4f}")

    # ── Plots ─────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{exercise} [{split}] — Per-class Binary Classifiers", fontweight="bold")

    ax = axes[0]
    ax.scatter(preds["true_scores"], preds["derived_scores"], alpha=0.4, s=15)
    lo = min(preds["true_scores"].min(), preds["derived_scores"].min())
    hi = max(preds["true_scores"].max(), preds["derived_scores"].max())
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
    ax.set_xlabel("True Score"); ax.set_ylabel("Derived Score")
    ax.set_title(f"Score Correlation  (ρ={rho:.3f})")

    ax2 = axes[1]
    aucs = []
    for i in range(len(error_names)):
        try:
            aucs.append(roc_auc_score(preds["true_labels"][:, i], preds["pred_probs"][:, i]))
        except ValueError:
            aucs.append(float("nan"))
    bars = ax2.bar(error_names, aucs, color="steelblue")
    ax2.axhline(0.5, color="red", linestyle="--", linewidth=1, label="random")
    ax2.set_ylim(0, 1); ax2.set_ylabel("ROC-AUC")
    ax2.set_title("Per-class AUC"); ax2.legend()
    for bar, v in zip(bars, aucs):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{v:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout(); plt.show()

    return {
        "macro_f1":     macro_f1,
        "spearman_rho": rho,
        "mae_score":    mae,
        "preds":        preds,
        "error_names":  error_names,
    }


print("Per-class binary evaluation functions defined (predict_all_binary, evaluate_binary).")

In [ ]:
"""
Run evaluation for all trained per-class binary classifiers on the test split.
Models are moved to GPU temporarily for inference, then back to CPU.
"""
import gc
all_results = {}
eval_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for exercise in ["OHP", "Squat", "BarbellRow"]:
    print(f"\nEvaluating {exercise}…")
    error_names = list(EXERCISE_CONFIG[exercise]["label_files"].keys())

    # Move models to GPU for eval
    models_dict     = {}
    thresholds_dict = {}
    for cname in error_names:
        model = trained_models[exercise][cname].to(eval_device)
        models_dict[cname] = model
        thresholds_dict[cname] = histories[exercise][cname].get("best_threshold", 0.5)

    # Test loader (full multi-label dataset, no balancing)
    test_loader = make_dataloader(
        "test", exercise, ALL_LABELS[exercise], ALL_SPLITS[exercise],
        batch_size=64, fixed_len=cfg.fixed_len,
        use_angles=cfg.use_angles, use_velocity=cfg.use_velocity,
        use_weighted_sampler=False, num_workers=4,
    )

    results = evaluate_binary(
        models_dict, test_loader, exercise=exercise,
        split="test", thresholds_dict=thresholds_dict,
    )
    all_results[exercise] = results

    # Move models back to CPU to free GPU
    for cname in error_names:
        trained_models[exercise][cname] = models_dict[cname].cpu()
    torch.cuda.empty_cache()
    gc.collect()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n\n" + "="*55)
print(f"  {'Exercise':<15}  {'Macro F1':>9}  {'Spearman ρ':>10}  {'MAE':>7}")
print("="*55)
for ex, res in all_results.items():
    print(f"  {ex:<15}  {res['macro_f1']:>9.4f}  {res['spearman_rho']:>10.4f}  {res['mae_score']:>7.4f}")

---
## Model Performance Summary

Per-class binary classification results on the **test split**.  
Each error class is predicted by an independent binary model (ExerciseTCN, hidden=128, layers=6).  
Thresholds were tuned on the validation set during training.

In [52]:
"""
Performance summary table — per-class metrics + exercise-level aggregates.
Requires: all_results, histories (from evaluation cell above).
"""
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ── Per-class metrics table ───────────────────────────────────────────────────
header = (f"  {'Exercise':<12} {'Error Class':<20} {'Prec':>6} {'Recall':>6} "
          f"{'F1':>6} {'AUC':>6} {'Thr':>5}  {'#Pos':>5} {'#Neg':>5}")
sep    = "─" * len(header)

print("Per-class Binary Classification Results (Test Set)")
print(sep)
print(header)
print(sep)

for exercise in ["OHP", "Squat", "BarbellRow"]:
    res         = all_results[exercise]
    preds       = res["preds"]
    error_names = res["error_names"]

    for i, ename in enumerate(error_names):
        prec = precision_score(preds["true_labels"][:, i], preds["pred_labels"][:, i], zero_division=0)
        rec  = recall_score   (preds["true_labels"][:, i], preds["pred_labels"][:, i], zero_division=0)
        f1   = f1_score       (preds["true_labels"][:, i], preds["pred_labels"][:, i], zero_division=0)
        try:
            auc = roc_auc_score(preds["true_labels"][:, i], preds["pred_probs"][:, i])
        except ValueError:
            auc = float("nan")
        thr   = histories[exercise][ename].get("best_threshold", 0.5)
        n_pos = int(preds["true_labels"][:, i].sum())
        n_neg = int(len(preds["true_labels"][:, i]) - n_pos)

        print(f"  {exercise:<12} {ename:<20} {prec:>6.3f} {rec:>6.3f} "
              f"{f1:>6.3f} {auc:>6.3f} {thr:>5.2f}  {n_pos:>5d} {n_neg:>5d}")

    print(sep)

# ── Exercise-level summary ────────────────────────────────────────────────────
print(f"\n{'Exercise-Level Summary':^{len(header)}}")
print(sep)
print(f"  {'Exercise':<15}  {'Macro F1':>9}  {'Spearman ρ':>10}  {'MAE':>7}  {'#Test':>6}")
print(sep)
for exercise in ["OHP", "Squat", "BarbellRow"]:
    res = all_results[exercise]
    n   = len(res["preds"]["derived_scores"])
    print(f"  {exercise:<15}  {res['macro_f1']:>9.4f}  {res['spearman_rho']:>10.4f}  "
          f"{res['mae_score']:>7.4f}  {n:>6d}")
print(sep)

Per-class Binary Classification Results (Test Set)
──────────────────────────────────────────────────────────────────────────────────
  Exercise     Error Class            Prec Recall     F1    AUC   Thr   #Pos  #Neg
──────────────────────────────────────────────────────────────────────────────────
  OHP          error_elbows          0.254  1.000  0.405  0.656  0.05     86   253
  OHP          error_knees           0.713  0.602  0.653  0.802  0.55    128   211
──────────────────────────────────────────────────────────────────────────────────
  Squat        knees_inward          0.268  0.306  0.286  0.621  0.55     36   207
  Squat        knees_forward         0.693  0.994  0.817  0.630  0.20    168    75
  Squat        shallow_depth         0.284  1.000  0.442  0.515  0.05     69   174
──────────────────────────────────────────────────────────────────────────────────
  BarbellRow   lumbar_error          0.457  0.272  0.341  0.655  0.55    390  1801
  BarbellRow   torso_angle          

---
## Prediction Examples with MediaPipe Landmark Overlays

For each error class we show examples of **correct predictions** on the test set:
- **True Positive** — the model correctly detected the error (ground-truth = 1, prediction = 1)
- **True Negative** — the model correctly identified a clean rep (ground-truth = 0, prediction = 0)

Video frames (OHP / Squat) or images (BarbellRow) are overlaid with the full MediaPipe 33-landmark skeleton.

In [53]:
"""
Visualise correct predictions with MediaPipe landmark skeleton overlays.
For each error class: 1 True Positive + 1 True Negative example.

OHP / Squat  → extract middle frame from video (.mp4)
BarbellRow   → load image directly (.jpg)
Landmarks    → loaded from pose .npy file, drawn as skeleton on the frame.
"""
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── MediaPipe 33-landmark skeleton connections ─────────────────────────────────
# Standard POSE_CONNECTIONS (pairs of landmark indices to draw as lines)
POSE_CONNECTIONS = [
    # Face
    (0, 1), (1, 2), (2, 3), (3, 7),   # left eye → left ear
    (0, 4), (4, 5), (5, 6), (6, 8),   # right eye → right ear
    (9, 10),                           # mouth
    # Torso
    (11, 12),                          # shoulder to shoulder
    (11, 23), (12, 24),               # shoulders to hips
    (23, 24),                          # hip to hip
    # Left arm
    (11, 13), (13, 15),               # shoulder → elbow → wrist
    (15, 17), (15, 19), (15, 21),     # wrist → pinky, index, thumb
    (17, 19),
    # Right arm
    (12, 14), (14, 16),
    (16, 18), (16, 20), (16, 22),
    (18, 20),
    # Left leg
    (23, 25), (25, 27),               # hip → knee → ankle
    (27, 29), (27, 31), (29, 31),     # ankle → heel, foot_index
    # Right leg
    (24, 26), (26, 28),
    (28, 30), (28, 32), (30, 32),
]

# Colour palette for different body regions
def _connection_color(i, j):
    """Return BGR colour for a connection based on body part."""
    torso  = {11, 12, 23, 24}
    l_arm  = {11, 13, 15, 17, 19, 21}
    r_arm  = {12, 14, 16, 18, 20, 22}
    l_leg  = {23, 25, 27, 29, 31}
    r_leg  = {24, 26, 28, 30, 32}
    pair   = {i, j}
    if pair <= torso:  return (255, 200, 0)    # cyan-ish torso
    if pair <= l_arm:  return (0, 255, 128)    # green left arm
    if pair <= r_arm:  return (0, 128, 255)    # orange right arm
    if pair <= l_leg:  return (255, 0, 128)    # magenta left leg
    if pair <= r_leg:  return (128, 0, 255)    # purple right leg
    return (200, 200, 200)                     # grey fallback (face)


def draw_landmarks_on_frame(frame, landmarks_2d, landmark_radius=4, line_thickness=2):
    """
    Draw MediaPipe pose landmarks and skeleton on a BGR frame.

    Parameters
    ----------
    frame        : (H, W, 3) BGR uint8 image
    landmarks_2d : (33, 2) or (33, 3) — normalised [x, y, (z)]
                   x ∈ [0,1] maps to width, y ∈ [0,1] maps to height
    """
    h, w = frame.shape[:2]
    annotated = frame.copy()

    # Convert normalised coords to pixel coords
    pts = []
    for lm in landmarks_2d:
        px = int(lm[0] * w)
        py = int(lm[1] * h)
        pts.append((px, py))

    # Draw connections (lines)
    for (i, j) in POSE_CONNECTIONS:
        if np.isnan(landmarks_2d[i]).any() or np.isnan(landmarks_2d[j]).any():
            continue
        color = _connection_color(i, j)
        cv2.line(annotated, pts[i], pts[j], color, line_thickness, cv2.LINE_AA)

    # Draw landmark dots
    for idx, (px, py) in enumerate(pts):
        if np.isnan(landmarks_2d[idx]).any():
            continue
        # Larger dots for major joints
        major_joints = {11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28}
        r = landmark_radius + 2 if idx in major_joints else landmark_radius
        cv2.circle(annotated, (px, py), r, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(annotated, (px, py), r, (0, 0, 0), 1, cv2.LINE_AA)

    return annotated


def load_frame_for_rep(exercise, rep_key):
    """
    Load a display frame for the given rep.
    OHP / Squat → middle frame from .mp4 video.
    BarbellRow  → the .jpg image directly.
    Returns: BGR frame (H, W, 3) or None.
    """
    if exercise == "BarbellRow":
        img_path = BARBELL_IMAGES_DIR / f"{rep_key}.jpg"
        if img_path.exists():
            return cv2.imread(str(img_path))
        return None
    else:
        vid_dir  = video_dir_for_exercise(exercise)
        vid_path = vid_dir / f"{rep_key}.mp4"
        if not vid_path.exists():
            return None
        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            return None
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        mid = n_frames // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
        ret, frame = cap.read()
        cap.release()
        return frame if ret else None


def load_landmarks_for_rep(exercise, rep_key):
    """
    Load raw pose landmarks for a rep.
    Returns: (T, 33, 3) numpy array or None.
    """
    pose_path = POSE_ROOT / exercise / f"{rep_key}.npy"
    if pose_path.exists():
        return np.load(pose_path)
    return None


# ── Find correct prediction examples ──────────────────────────────────────────
def find_correct_examples(exercise, class_idx, preds, n_tp=1, n_tn=1):
    """
    Find rep_keys for correct TP and TN predictions.
    Returns: {"tp": [keys], "tn": [keys]}
    """
    true   = preds["true_labels"][:, class_idx]
    pred   = preds["pred_labels"][:, class_idx]
    keys   = preds["rep_keys"]

    tp_mask = (true == 1) & (pred == 1)
    tn_mask = (true == 0) & (pred == 0)

    # Shuffle to get varied examples each run
    rng = np.random.RandomState(42)
    tp_indices = np.where(tp_mask)[0]; rng.shuffle(tp_indices)
    tn_indices = np.where(tn_mask)[0]; rng.shuffle(tn_indices)

    # Filter to those that have both a video/image AND a pose file
    def _has_data(idx):
        rk = keys[idx]
        frame = load_frame_for_rep(exercise, rk)
        lm    = load_landmarks_for_rep(exercise, rk)
        return frame is not None and lm is not None

    tp_keys = [keys[i] for i in tp_indices if _has_data(i)][:n_tp]
    tn_keys = [keys[i] for i in tn_indices if _has_data(i)][:n_tn]
    return {"tp": tp_keys, "tn": tn_keys}


# ── Main visualisation loop ───────────────────────────────────────────────────
for exercise in ["OHP", "Squat", "BarbellRow"]:
    res         = all_results[exercise]
    preds       = res["preds"]
    error_names = res["error_names"]
    n_classes   = len(error_names)

    fig, axes = plt.subplots(n_classes, 2, figsize=(10, 5 * n_classes))
    if n_classes == 1:
        axes = axes[np.newaxis, :]  # ensure 2D array
    fig.suptitle(f"{exercise} — Correct Prediction Examples with Pose Overlay",
                 fontsize=14, fontweight="bold", y=1.01)

    for ci, ename in enumerate(error_names):
        examples = find_correct_examples(exercise, ci, preds, n_tp=1, n_tn=1)

        for col, (label_type, key_list) in enumerate([("True Positive", examples["tp"]),
                                                       ("True Negative", examples["tn"])]):
            ax = axes[ci, col]
            if not key_list:
                ax.text(0.5, 0.5, f"No {label_type}\nexample found",
                        ha="center", va="center", fontsize=12, transform=ax.transAxes)
                ax.set_title(f"{ename} — {label_type}")
                ax.axis("off")
                continue

            rep_key = key_list[0]
            frame   = load_frame_for_rep(exercise, rep_key)
            lm_seq  = load_landmarks_for_rep(exercise, rep_key)

            if exercise == "BarbellRow":
                lm_frame = lm_seq[0]  # single frame (1, 33, 3) → (33, 3)
            else:
                # Pick the middle frame landmarks to match the video frame
                mid_idx = lm_seq.shape[0] // 2
                lm_frame = lm_seq[mid_idx]

            annotated = draw_landmarks_on_frame(frame, lm_frame)
            # Convert BGR → RGB for matplotlib
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

            ax.imshow(annotated_rgb)
            status = "Error present" if label_type == "True Positive" else "Clean rep"
            prob_val = preds["pred_probs"][np.where(preds["rep_keys"] == rep_key)[0][0], ci]
            ax.set_title(f"{ename} — {label_type}\n{status} | prob={prob_val:.2f} | key={rep_key}",
                         fontsize=10)
            ax.axis("off")

    plt.tight_layout()
    plt.show()
    print()

print("✓  Visualisation complete — showing TP/TN examples for all 7 error classes.")


✓  Visualisation complete — showing TP/TN examples for all 7 error classes.


---
## Inference Pipeline

Given a **new video** of a single rep:
1. Extract MediaPipe pose `(T, 33, 3)`
2. Normalise → build feature vector `(T, F)`
3. Interpolate to `(100, F)`
4. Run each per-class binary model → error probabilities
5. Derive quality score as `1 - mean(error_probs)`
6. Map error flags to human-readable feedback

The `AQAPredictor` class loads per-class checkpoints and exposes `.predict(video_path)`.

In [ ]:
"""
AQAPredictor: end-to-end inference from a raw video file.
Loads per-class binary model checkpoints and combines predictions.
"""
import torch
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass


# ── Human-readable feedback templates ─────────────────────────────────────────
FEEDBACK_TEMPLATES: Dict[str, Dict[str, str]] = {
    "OHP": {
        "error_elbows": (
            "Elbow flare detected — keep elbows slightly forward of the bar, "
            "not flared out to the sides, to protect the shoulder joint."
        ),
        "error_knees": (
            "Knee bend detected during the press — lock your knees and brace "
            "your legs throughout the lift to maintain a stable base."
        ),
    },
    "Squat": {
        "knees_inward": (
            "Knee cave (valgus collapse) detected — drive your knees out "
            "in line with your toes throughout the descent and ascent."
        ),
        "knees_forward": (
            "Excessive forward knee travel — shift weight to heels and "
            "push hips back to reduce anterior knee stress."
        ),
        "shallow_depth": (
            "Shallow squat depth — aim to break parallel (hip crease below "
            "knee) for full range of motion and posterior chain engagement."
        ),
    },
    "BarbellRow": {
        "lumbar_error": (
            "Lumbar rounding detected — maintain a neutral spine by "
            "hinging at the hips and engaging your lower back and core."
        ),
        "torso_angle": (
            "Torso angle too upright — aim for ~45 degree forward lean to "
            "maximise lat engagement and reduce lower back strain."
        ),
    },
}


@dataclass
class AQAPrediction:
    exercise:    str
    video_path:  str
    score:       float            # derived quality in [0, 1]
    grade:       str              # A / B / C / D / F
    errors:      List[str]        # error names flagged
    feedback:    List[str]        # human-readable feedback strings
    probs:       Dict[str, float] # error probability per class

    def __str__(self):
        lines = [
            f"Exercise : {self.exercise}",
            f"Video    : {self.video_path}",
            f"Score    : {self.score:.2f}  [{self.grade}]",
            "Errors   :",
        ]
        if self.errors:
            for fb in self.feedback:
                lines.append(f"  {fb}")
        else:
            lines.append("  No errors detected — clean rep!")
        return "\n".join(lines)


def _score_to_grade(score: float) -> str:
    if score >= 0.90: return "A"
    if score >= 0.75: return "B"
    if score >= 0.60: return "C"
    if score >= 0.40: return "D"
    return "F"


class AQAPredictor:
    """
    End-to-end inference predictor using per-class binary models.

    Loads one checkpoint per error class from the checkpoint directory.
    Combines their predictions into a multi-label output.

    Parameters
    ----------
    exercise       : 'OHP', 'Squat', or 'BarbellRow'
    checkpoint_dir : directory containing {exercise}_{class_name}_best.pt files
    device         : 'auto', 'cpu', 'cuda'
    """

    def __init__(
        self,
        exercise:       str,
        checkpoint_dir: Path = CHECKPOINT_DIR,
        device:         str  = "auto",
    ):
        self.exercise = exercise
        self.device   = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        ) if device == "auto" else torch.device(device)

        error_names = list(EXERCISE_CONFIG[exercise]["label_files"].keys())
        self.error_names  = error_names
        self.models       = {}
        self.thresholds   = {}

        for cname in error_names:
            ckpt_path = checkpoint_dir / f"{exercise}_{cname}_best.pt"
            if not ckpt_path.exists():
                raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

            ckpt = torch.load(ckpt_path, map_location=self.device, weights_only=False)
            model = ExerciseTCN(
                input_dim=cfg.feature_dim, n_error_classes=1,
                hidden_dim=cfg.hidden_dim, n_layers=cfg.n_layers,
                kernel_size=3, dropout=cfg.dropout,
            )
            model.load_state_dict(ckpt["model_state"])
            model.to(self.device).eval()
            self.models[cname]     = model
            self.thresholds[cname] = ckpt["history"].get("best_threshold", 0.5)

        self.extractor = PoseExtractor() if MP_AVAILABLE else None
        print(f"AQAPredictor ready for {exercise}: {len(self.models)} error models loaded")

    @torch.no_grad()
    def predict(self, video_path: Path) -> AQAPrediction:
        """
        Run full inference on a video file.

        Steps:
          1. Extract pose landmarks (T, 33, 3) with MediaPipe
          2. Build feature vector (T, F)
          3. Interpolate to fixed length (100, F)
          4. Run each per-class binary model
          5. Combine predictions and derive quality score
        """
        video_path = Path(video_path)

        # 1. Extract pose
        if self.extractor is None:
            raise RuntimeError("MediaPipe not installed — cannot extract pose.")
        landmarks = self.extractor.extract(video_path)
        if landmarks is None:
            raise RuntimeError(f"Could not read video: {video_path}")

        # 2. Build feature vector
        features = build_feature_vector(
            landmarks,
            use_angles=cfg.use_angles,
            use_velocity=cfg.use_velocity,
        )

        # 3. Interpolate to fixed length
        features = interpolate_sequence(features, cfg.fixed_len)

        # 4. Run each binary model
        x = torch.from_numpy(features).unsqueeze(0).to(self.device)
        probs = {}
        for cname in self.error_names:
            logits = self.models[cname](x)  # (1, 1)
            prob   = torch.sigmoid(logits).item()
            probs[cname] = prob

        # 5. Combine predictions
        flagged_errors = [
            cn for cn in self.error_names
            if probs[cn] >= self.thresholds[cn]
        ]
        feedback = [
            FEEDBACK_TEMPLATES.get(self.exercise, {}).get(e, f"Error detected: {e}")
            for e in flagged_errors
        ]
        score = 1.0 - np.mean(list(probs.values()))

        return AQAPrediction(
            exercise   = self.exercise,
            video_path = str(video_path),
            score      = float(score),
            grade      = _score_to_grade(float(score)),
            errors     = flagged_errors,
            feedback   = feedback,
            probs      = probs,
        )

    def __del__(self):
        if hasattr(self, "extractor") and self.extractor:
            self.extractor.close()


# ── Usage example ─────────────────────────────────────────────────────────────
print("Inference pipeline ready.")
print()
print("Usage:")
print("  predictor = AQAPredictor('OHP')")
print("  result    = predictor.predict('/path/to/overhead_press.mp4')")
print("  print(result)")


---
## Pipeline Summary

### Execution Order (top to bottom)

1. **Config cell** — set paths, hyperparameters, seed
2. **Label loading** — loads all 7 error labels across 3 exercises
3. **Pose extraction** — run once, skip if `.npy` files already exist
4. **Normalisation** — run once, skip if processed files exist
5. **Training cell** — trains 7 binary models (or loads from checkpoints)
6. **Evaluation cell** — computes per-class F1/AUC and derived score metrics
7. **Inference** — `AQAPredictor('OHP').predict('video.mp4')`

### Key Design Decisions

| Decision | Rationale |
|----------|-----------|
| **Per-class binary models** | Eliminates complex imbalance correction (no pos_weight, focal loss, or WeightedRandomSampler) |
| **Balanced 1:1 datasets** | Minority oversampled + augmentation makes repeated samples appear different |
| **Classification-only TCN** | Score derived post-hoc; avoids dual-head loss balancing |
| **Subject-level splits** | Prevents data leakage (same subject never in train + test) |
| **Frame→rep aggregation** | shallow_depth labels use 3-part frame keys; aggregated to 2-part rep keys |
| **AttentionPool** | Learns to focus on error-relevant frames rather than uniform average |

### Potential Pitfalls

| Risk | Mitigation |
|------|-----------|
| **Subject leakage** | Always split by subject ID, verified by `verify_subject_splits()` |
| **Class imbalance** | Per-class balanced 1:1 datasets with augmented oversampling |
| **NaN landmarks** | `fill_nan_frames()` interpolates missing detections |
| **Variable rep speed** | Interpolation to fixed 100 frames normalises duration |
| **Overfitting** | Early stopping + dropout + weight decay + data augmentation |